In [ ]:
%pip -q install hmmlearn

# RegDet V1.1 — the SIDEWAYS over-occupancy fix

## The complaint

`SIDEWAYS` occupancy is far too high. Measured on the user's real 2h data:
**34.7–35.7%** at the shipped config, **41.7%** on the daily-fit arm.
`PROJECT_STATE.md` records an earlier run at **19.8%**. Target: back to roughly
**20%** (accept **18–25%**) *without breaking anything else*.

## The diagnosis I was handed — and the answer

> *"V1.1 emits SIDEWAYS via an ABSOLUTE-MAJORITY rule. `CONF_L = 0.50` on a
> THREE-way partition demands an absolute majority. A posterior of
> bull 0.45 / neutral 0.30 / bear 0.25 is an unambiguous bull reading and is
> forced to SIDEWAYS."*

**This is refuted.** Section 4 measures it directly. `CONF_L = 0.50` forces
`SIDEWAYS` on **0.1%** of bars. Turning the override off entirely — pure MAP,
`CONF_L = 0` — moves `SIDEWAYS` by **0.05 percentage points**. The theorised
`0.45 / 0.30 / 0.25` posterior essentially never occurs: the median winning
bucket mass is **0.99**, the 5th percentile is **0.59**, and only 0.2% of bars
sit below 0.50 at all. A filtered posterior from a well-separated 5-state HMM is
close to one-hot, and bucket aggregation (2 bull states + 2 bear states + 1 side
state) pushes the winner higher still. There is no absolute-majority tax to
remove.

**What actually drives it:** the `SIDE` bucket *genuinely wins the argmax* on
~29% of bars. `direction_buckets` gives N=5 a split of **2 bear / 1 side / 2
bull**, so one state in five is the sideways state — but that state is the
*quiet, modal* market state, and it collects far more than a fifth of the bars.
The rule is already MAP. MAP is the problem, not the cure.

**Consequence for the fix:** every candidate that only makes the *directional*
buckets work harder (`R1`–`R5`) is structurally incapable of lowering
`SIDEWAYS`, and the sweeps confirm it — several of them *raise* it. The fix has
to change **what `SIDE` has to prove**, which is what `R6`/`R10` do.

## The anti-cheat requirement, and why the headline metric needed a companion

Cutting `SIDEWAYS` is trivial and worthless if the newly-directional bars are
noise. The declared audit is

> **TREND EFFICIENCY DISCRIMINATION** = median trailing trend efficiency on
> directional bars − median on `SIDEWAYS` bars.

This is legitimate and non-circular: `trend_efficiency` feeds the **INTENSITY
(H/L)** axis of the engine, never the **direction** axis, so using it to audit a
*direction* rule is an independent test. It is also strictly causal — a trailing
window, bars ≤ *t* only.

It is, however, a **difference of medians between two pools whose composition
changes**, and that makes it partly an artifact when 9pp of bars move from one
pool to the other: the promoted bars sit *between* the two medians by
construction, so they drag the directional median down even when the rule sorted
them correctly. Two companions are therefore reported alongside it — both
causal, both forward-return-free:

* **AUC** = P(efficiency of a random directional bar > efficiency of a random
  `SIDEWAYS` bar), a rank statistic immune to the pool-composition artifact;
* the **ORDERING TEST**: median efficiency of *(a)* bars directional under both
  baseline and candidate, *(b)* bars **moved** `SIDEWAYS`→directional, *(c)*
  bars `SIDEWAYS` under both. A genuine rule needs **a > b > c** — the bars it
  promoted must be more directional than the ones it left behind.
* a **RANDOM-PROMOTION CONTROL**: promote the *same number* of `SIDEWAYS` bars
  at random. Any candidate that cannot beat this is relabelling noise, by
  definition.

The headline gap is reported unmodified, shrink and all. The companions explain
it; they do not replace it.

## Everything else is measured too

`S` (seed-set label agreement, R=4 disjoint sets), `L` (ZigZag transition lag),
`W` (switches/100) and guards `G1`–`G4`, reused verbatim from
`build_stability_lag.py`. `W` is the key co-guard: bars parked in `SIDEWAYS` now
flip between `BULL` and `BEAR`, so cutting `SIDEWAYS` should raise `W`. Baseline
is row 1 of every table. **No composite score.**

## 1. Engine, inlined verbatim

Harvested from `build_master_notebook_v2.py cell 3 (constants/imports/CONFIGS), cell 5 (synthetic 2h generator), cell 7 (feature + labeling engine), cell 9 (regime-background plot helpers)`. Nothing in the labeling core is edited: the
candidate rules act through a documented mass-override hook whose OFF value is
asserted **bit-for-bit** against the untouched engine in section 3.

In [ ]:
# ==========================================================================
# ENGINE CELL 1/4 -- constants, imports, CONFIGS (VERBATIM)
# ==========================================================================
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt          # default backend -> inline figures
import matplotlib.patches as mpatches
import matplotlib.dates as mdates        # date2num for the batched regime bands
from hmmlearn import hmm
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

# ---------------------------------------------------------------------------
# Walk-forward harness config (matches capacity_ladder.py / the head-to-head)
# ---------------------------------------------------------------------------
HMM_ITER       = 2000          # EM iteration cap (fits converge in ~75-240 iters, <1s each)
N_FOLDS        = 4             # anchored walk-forward test blocks
MIN_TRAIN_FRAC = 0.50          # first fold trains on >= this fraction of bars
BARS_PER_DAY   = 3             # 2h bars per NSE session
LOOKBACK_SCALE = 1.0           # feature-window multiplier (1.0 = production windows)

# ---------------------------------------------------------------------------
# LABELING KNOBS — verbatim from REGDET_CONFIG in regdet_v11.py.
# These drive the direction + intensity + chop-filter scheme used EVERYWHERE in
# this notebook (walk-forward signal, regime overlay, evaluation suite).
# ---------------------------------------------------------------------------
CONF_L         = 0.50   # low-confidence override: if the WINNING direction bucket's
                        # aggregated probability mass is below this, the bar is forced
                        # to SIDEWAYS. Higher = more SIDEWAYS, fewer directional calls.
Z_HI           = 0.5    # trend-MAGNITUDE gate: |trend_z| must reach this for a bull/bear
                        # bar to escalate to H_BULL/H_BEAR. trend_z is TREND_FEATURE
                        # standardized against a mu/sd baseline FROZEN on the fit window
                        # (causal — never recomputed from bars the model has not seen).
                        # Lower = H fires more often (and flickers more).
EFF_HI         = 0.35   # trend-EFFICIENCY gate = THE CHOP FILTER. Efficiency is
                        # |net move| / |total path walked| over EFF_WIN bars: ~1.0 in a
                        # clean trend, ~0.0 in a range that keeps doubling back. Magnitude
                        # alone cannot separate a real trend from a swing inside a bracket
                        # (both show big momentum); this can. Requiring BOTH gates keeps
                        # range-bound swings at L_BULL/L_BEAR. Raise = stricter (less H).
EFF_WIN        = 9      # bars over which efficiency is measured (~ TREND_FEATURE horizon)
TREND_FEATURE  = 'mom_3d'   # the feature whose z-score grades trend magnitude

# ---------------------------------------------------------------------------
# LABELING FIXES 1-3 (see Section 7H for the measured A/B). Each is behind its
# own switch, and each switch has a value that reproduces the OLD behaviour
# bit-for-bit, so their effects can be separated and audited.
# ---------------------------------------------------------------------------

# FIX 1 -- DIRECTION SCORER: features that are excluded from the composite
# BULLISHNESS score. vol_2h and vol_expansion are MAGNITUDE measures: they say
# how big moves are, not which way they point. Feeding them into a DIRECTION
# score is a category error -- with the old weights the risk block (vol_2h,
# vol_expansion, vix_chg, drawdown; total 4.0) outweighed the directional block
# (ret_2h, mom_1d/3d/5d, dist_ma; total 3.2), so a violent RALLY (high vol, high
# vol expansion, still-elevated drawdown) scored BEARISH while price ripped up.
# These two features REMAIN HMM INPUT FEATURES -- they are genuinely informative
# about state -- they simply stop voting on direction. drawdown and vix_chg keep
# their -1.0 direction weight: both are defensibly signed (a deeper drawdown and
# a rising VIX really are bearish, not merely "big").
# DIRECTION_EXCLUDE = () reproduces the old scorer exactly.
DIRECTION_EXCLUDE = ('vol_2h', 'vol_expansion')

# FIX 2 -- LABEL HYSTERESIS (causal confirmation delay): a candidate new
# DIRECTION must persist for CONFIRM_BARS consecutive bars before the emitted
# label is allowed to flip; until then the previous emitted direction is held.
# This is a DELAY, not a smoother: bar t's emitted label is a function of bars
# <= t only. (An earlier version of this project shipped a "smoother" that
# decided whether to erase a run by inspecting the run's full REALIZED length --
# that is look-ahead and was removed. Nothing of that shape is reintroduced here;
# the causality probe in 7.0 re-proves it under hysteresis.)
# CONFIRM_BARS = 1 reproduces the old behaviour exactly.
CONFIRM_BARS   = 2

# FIX 3 -- GATE HYSTERESIS: the H escalation gates become enter/exit BANDS.
# Escalate L -> H when |z| >= Z_HI AND eff >= EFF_HI (unchanged), but only
# de-escalate H -> L when |z| < Z_HI_EXIT OR eff < EFF_HI_EXIT. Without this,
# bars sitting near a single threshold flicker H->L->H->L every bar.
# Z_HI_EXIT = Z_HI and EFF_HI_EXIT = EFF_HI reproduces the old behaviour exactly.
Z_HI_EXIT      = 0.35
EFF_HI_EXIT    = 0.25

# FIX 4 -- PER-BAR DIRECTION (this is the architectural one).
#
# THE FLAW IT ADDRESSES. Until now DIRECTION was a per-STATE property: the HMM
# assigns bar t a distribution over states, each STATE is bucketed bear/side/bull
# ONCE from the rank of its composite mean profile, and the bar inherits its
# state's direction. But an HMM state is directionally MIXED. Volatility is
# direction-agnostic, so the model reliably learns a "violent" state that
# contains BOTH sharp selloffs AND sharp rallies. ONE bucket label for that state
# cannot be right for every bar in it, no matter how the bucket is chosen.
#
# Real-data evidence (2874 2h Nifty bars) on the 366 causal high-vol RALLY bars
# (top vol_2h quartile AND trailing mom_3d > 0):
#   baseline          46.2% bear / 7.4% sideways / 46.4% bull
#   + FIX 1 (no vol)  11.2% bear / 42.3% sideways / 46.4% bull
# The red became GREY, not green -- bull did not move at all -- and SIDEWAYS then
# showed the HIGHEST forward return of any label (+0.717% at 15 bars vs H_BULL
# +0.026%), which is exactly what "a bullish population got parked in SIDEWAYS"
# looks like. FIX 1 removed a wrong vote; it could not add a right one, because
# the vote is cast once per state and not once per bar.
#
# THE FIX. Add a CAUSAL PER-BAR directional score from the SIGNED features only
# (ret_2h, mom_1d, mom_3d, mom_5d, dist_ma -- reusing FEATURE_SIGN/FEATURE_MAG),
# each standardized against a mu/sd baseline FROZEN on the fit window exactly the
# way trend_z is, then map it to three per-bar direction masses through a softmax
# and BLEND those with the state-level masses:
#
#     mass = (1 - BAR_DIR_WEIGHT) * state_mass + BAR_DIR_WEIGHT * bar_mass
#
# A convex combination of two points on the 3-simplex is on the 3-simplex, so
# bull+side+bear == 1 still holds bar by bar and every downstream mechanism --
# the prob_* partition, tactical_regime_confidence, the CONF_L override, the
# intensity gates, the CONFIRM_BARS hysteresis -- is untouched. The HMM keeps
# supplying market character, persistence and the confidence signal; only the
# DIRECTION ATTRIBUTION moves from per-state to per-bar.
#
# vol_2h / vol_expansion are structurally excluded from this score: they are
# magnitude, not direction. That is asserted, not merely intended.
#
# BAR_DIR_WEIGHT = 0.0 reproduces the pre-fix behaviour BIT-FOR-BIT -- the blend
# degenerates to 1.0*state_mass + 0.0*bar_mass, which is exact in IEEE754 for
# non-negative masses. Section 7H asserts that equality rather than assuming it.
# BAR_DIR_WEIGHT = 1.0 decides direction purely per bar (the HMM then contributes
# only character/persistence, not direction). Section 7H sweeps 0/0.25/0.5/0.75/1.
#
# ADOPTED VALUE 0.0 -- FIX 4 IS RETAINED AS A SWITCH BUT SET OFF.
#
# It was 0.75 (and before that 0.5). It is now 0.0. The machinery, the sweep in
# 7H-vi and the overlay in 7H-vii all stay; only the shipped weight moved.
#
# WHY IT WAS TURNED OFF. Scored across three real-data runs, the benefit fix 4 was
# added for -- more bull on high-volatility rally bars -- did NOT reproduce:
# +19.7 pp once, +1.4 pp the second time, and on the third the rally gain came
# from fixes 1-2 with fix 4 already OFF. The COST reproduced every time: it broke
# the direction-level forward-return ordering. A controlled A/B on the SAME data
# and the SAME fit, changing only w:
#
#     metric                        w = 0.75        w = 0.0
#     direction ordering 3/9/15     BROKEN/HOLDS/BROKEN   HOLDS/HOLDS/HOLDS
#     H_BULL fwd_3 HAC t            1.93 WARN       2.06 PASS
#     L_BULL fwd_9 HAC t            1.04 FAIL       2.10 PASS
#     BULL vs BEAR d @ fwd_3        0.006           0.118
#     H_BULL vs H_BEAR d @ fwd_3    0.139 WARN      0.214 PASS
#     strategy total return         +17.37%         +37.74%
#     strategy Sharpe               0.69            1.26
#
# w = 0 is where this project first produced HAC t > 2 on anything.
#
# The dead-zone note that justified 0.75 over 0.5 is still true and still the
# reason NOT to ship an intermediate value if fix 4 is ever switched back on: the
# filtered state posterior saturates near one-hot (confidence ~0.999 on most
# bars), so a convex blend cannot move the argmax until w > ~0.57. The choice is
# effectively between 0.0 (off) and >= ~0.75 (on); 0.25/0.5 are nominally on and
# behaviourally almost off, which is the worst of both.
#
# At w = 0.0 `dir_feats` is STILL passed everywhere it was before. The blend
# degenerates to 1.0*state_mass + 0.0*bar_mass -- exact in IEEE754 -- so labels
# are bit-identical to the no-dir_feats path (asserted in 7H), while
# `bar_dir_score` stays populated as a diagnostic column and the causality probe
# in Section 7.0 keeps testing it.
BAR_DIR_WEIGHT   = 0.0
BAR_DIR_FEATURES = ('ret_2h', 'mom_1d', 'mom_3d', 'mom_5d', 'dist_ma')
BAR_DIR_TAU      = 1.0   # softmax temperature on the per-bar z. The score is
                         # re-standardized on the fit window, so tau is in units
                         # of fit-window sd: |z| ~ 0.5*tau is where the leading
                         # direction's per-bar mass crosses 0.5. Lower tau =
                         # more decisive (more extreme) per-bar masses.

TRAIN_FRACTION = 0.70   # anchored fit fraction used by the PRODUCTION-style single fit
                        # in Section 7 (the engine's own TRAIN_FRACTION). The walk-forward
                        # in Section 5 uses MIN_TRAIN_FRAC/N_FOLDS fold edges instead.

# ---------------------------------------------------------------------------
# SEED ENSEMBLE — the identifiability fix (see Section 5d).
#
# hmm.GaussianHMM is fit by EM, a LOCAL optimizer. With one fixed seed the fit
# is not identified: refitting the same bars with different seeds lands in
# different local optima (train log-likelihood spread of ~1000 nats across 8
# seeds on the full-cov config) which segment the data differently. Multi-restart
# EM keeping the best log-likelihood does NOT fix it -- it collapses the LL
# spread but the surviving optima still disagree on the segmentation.
#
# What works instead: fit K models with K different seeds and average the
# DIRECTION-BUCKET PROBABILITY MASSES (bull / side / bear). Raw HMM state indices
# are arbitrary and permute freely between fits, so they cannot be averaged --
# but direction masses are permutation-INVARIANT semantic quantities, so they
# can. Each model's 3 masses sum to 1, so their average does too, and the
# labeling scheme downstream is bit-for-bit the same; only the SOURCE of the
# masses changes.
#
# ENSEMBLE_K = 1 reproduces the old single-fit behaviour exactly.
#
# Cost scales linearly in K (K fits per training slice). K=6 was the size the
# offline study measured (direction-call agreement between independent pools
# 81.9% single -> 91.9% at K=6), and K=6 is now the ADOPTED value: it is the size
# the stability study actually measured, and the cost is linear. The earlier K=4
# compromise existed only because Section 5d re-ran the ENTIRE walk-forward in
# both arms; 5d is OFF by default now (RUN_SEED_STABILITY=False below), so the
# runtime argument for K=4 no longer applies.
ENSEMBLE_K     = 6
BASE_SEED      = 42     # ensemble seeds are BASE_SEED + 0..K-1 (deterministic,
                        # so every run of this notebook is reproducible)

# ---------------------------------------------------------------------------
# RUNTIME KNOBS. These change ONLY how fast the notebook runs, never what it
# computes. Both have a value that reproduces the original code path exactly.
# ---------------------------------------------------------------------------

# N_JOBS -- ensemble fit parallelism. DEFAULT 1 (serial), and that default is
# deliberate. Read this before changing it.
#
# The K members of a seed ensemble ARE independent and each IS fully determined
# by its own random_state, so at the level of the algorithm, fitting them
# concurrently cannot change anything. It does anyway, for a reason that has
# nothing to do with this notebook's logic: MEASURED on this environment,
# `GaussianHMM.fit` is not bit-reproducible across OpenBLAS thread counts. The
# same seed on the same rows gives model parameters differing by ~1.5e-11
# between a 1-thread and a 4-thread BLAS, because threaded reductions sum in a
# different order and ~100-170 EM iterations amplify the last-bit difference.
# joblib's loky backend pins each worker to ONE inner thread (correctly -- it is
# avoiding oversubscription), so a parallel fit lands on the 1-thread arithmetic
# while the serial fit here lands on the multi-thread arithmetic.
#
# Measured, on 4 fits of 1500x9 at N=5 full-cov:
#     serial, default threads          3.19s   <- what this notebook does
#     serial, BLAS pinned to 1 thread  2.88s   params differ by 1.5e-11
#     parallel, 1 inner thread         2.48s   BIT-IDENTICAL to the line above
#     parallel, 4 inner threads       33.64s   10x SLOWER (oversubscription)
#
# So parallelism is available but only at 1 inner thread, and that arm is
# bit-identical to serial-at-1-thread -- NOT to serial-at-default-threads. The
# available speedup is ~1.3x on the fits, and the price is moving every model
# parameter in the 11th decimal. In a notebook where a 3-bar data perturbation
# has already flipped the config winner, that is a bad trade, so it is NOT the
# default. N_JOBS = 1 reproduces the historical numbers exactly.
#
# If you set N_JOBS != 1 you are choosing a different (equally valid, not more
# accurate) floating-point path, and the headline numbers may move slightly.
# Section 7.0 asserts and REPORTS this rather than hiding it.
N_JOBS         = 1

# Section 5d is a ONE-TIME IDENTIFIABILITY DIAGNOSTIC, not part of the pipeline:
# it re-runs the ENTIRE walk-forward 36 times (3 configs x 6 seeds x 2 arms) to
# ask whether the seed ensemble stabilised the config ranking. That question has
# been answered, and NOTHING downstream reads any variable it defines -- so on a
# normal run it is ~2/3 of the total wall time spent re-confirming a settled
# result. Default OFF. Set True to re-run it (e.g. after changing ENSEMBLE_K,
# N_STATES, the features or the folds -- any of which reopens the question).
RUN_SEED_STABILITY = False

CONFIDENCE_THRESHOLD_H  = 0.70   # chart reference line only
CONFIDENCE_THRESHOLD_L  = CONF_L # the actual SIDEWAYS override
HMM_PROB_DROP_THRESHOLD = 0.20   # confidence drop -> transition warning
VIX_SPIKE_THRESHOLD     = 0.25   # bar-over-bar VIX jump -> transition warning

REGIME_LABELS = ['H_BULL', 'L_BULL', 'SIDEWAYS', 'L_BEAR', 'H_BEAR']
REGIME_COLORS = {
    'H_BULL':   '#006400',
    'L_BULL':   '#90EE90',
    'SIDEWAYS': '#808080',
    'L_BEAR':   '#FFB6C1',
    'H_BEAR':   '#8B0000',
}

# Feature names follow regdet_v11.py exactly (ret_2h / vol_2h, not ret / vol).
FEATURE_COLS = ['ret_2h', 'mom_1d', 'mom_3d', 'mom_5d',
                'vol_2h', 'vol_expansion', 'vix_chg', 'drawdown', 'dist_ma']

# 5-feature primary subset from the feature-selection analysis (corr clustering
# + PCA + VIF): one representative per information family.
FEATURES_LEAN = ['ret_2h', 'mom_3d', 'vol_2h', 'vol_expansion', 'dist_ma']

BASE_WIN = dict(MOM_1D=1*BARS_PER_DAY, MOM_3D=3*BARS_PER_DAY, MOM_5D=5*BARS_PER_DAY,
                VOL_WIN=10, VOL_FAST=5, VOL_SLOW=20, SWING_WIN=20)

# Per-feature sign/magnitude weights for the subset-agnostic bullishness scorer.
#   raw (DIRECTION_EXCLUDE = ()):
#     score = ret_2h + 0.4*(m1+m3+m5) - vol_2h - vol_expansion - vix_chg - drawdown + dist_ma
#   with FIX 1 (DIRECTION_EXCLUDE = ('vol_2h','vol_expansion')) the two magnitude
#   terms drop out and the score becomes purely directional:
#     score = ret_2h + 0.4*(m1+m3+m5) - vix_chg - drawdown + dist_ma
# The exclusion is applied as a WEIGHT OF ZERO in `direction_weight` below, so it
# stays subset-agnostic: excluding a feature the subset does not contain is a
# no-op, and the HMM's own feature matrix is untouched.
FEATURE_SIGN = {
    'ret_2h': 1.0, 'mom_1d': 1.0, 'mom_3d': 1.0, 'mom_5d': 1.0, 'dist_ma': 1.0,
    'vol_2h': -1.0, 'vol_expansion': -1.0, 'vix_chg': -1.0, 'drawdown': -1.0,
}
FEATURE_MAG = {'mom_1d': 0.4, 'mom_3d': 0.4, 'mom_5d': 0.4}   # else 1.0

# ---------------------------------------------------------------------------
# FIX 5 -- INTENSITY_MODE: the SCALE-FREE H/L intensity gate.
#
# THE FLAW. H_BULL / H_BEAR escalate on a trend-MAGNITUDE gate
#     trend_z = (mom_3d - mu_fit) / sd_fit          |trend_z| >= Z_HI = 0.5
# with mu_fit / sd_fit frozen on the training window. Freezing them is correct
# for CAUSALITY -- but it makes the THRESHOLD meaningless. A constant threshold
# is only interpretable on a scale-free quantity, and trend_z is scale-free only
# if sd_fit happens to equal the CURRENT dispersion of mom_3d. Volatility
# clusters, so it never does: the gate's aggressiveness is governed by the ratio
# sd_fit / sd_now, which is an artefact of where the training cut was placed and
# has no economic meaning.
#
# Real-data evidence (the user's own run, same bars, same config -- ONLY the fit
# window differs):
#     fit 50%:  H_BULL 22.9%  L_BULL 16.2%  SIDEWAYS 36.0%  L_BEAR  8.3%  H_BEAR 16.5%
#     fit 70%:  H_BULL 21.8%  L_BULL 31.0%  SIDEWAYS 10.6%  L_BEAR 17.3%  H_BEAR 19.3%
# SIDEWAYS is 3.4x larger in one than the other. These are effectively two
# different detectors produced by an arbitrary backtest cut.
#
# THE FIX. trend_t = mom_3d / (vol_2h * sqrt(MOM_3D_BARS)) -- the t-statistic of
# the 9-bar move. mom_3d is the 9-bar sum of log returns; vol_2h is the trailing
# per-bar return sd. The ratio is DIMENSIONLESS and CONTEMPORANEOUS, and needs no
# fit-window baseline at all -- the fix REMOVES a fit-window dependence rather
# than relocating it. Both inputs are trailing rolling windows at bar t, so it is
# fully causal.
#
# A REJECTED alternative, measured and discarded: a trailing 250-bar quantile of
# |mom_3d|. It was WORSE than the status quo (H-firing spread 44.7 pp vs 35.8 pp)
# because a trailing window is a LAGGING scale estimate -- when volatility drops
# the window is full of stale high-vol bars and the H-rate collapsed to 11%.
#
# THE THRESHOLD. |trend_t| >= 0.5 would fire on ~68% of bars, so the threshold is
# not hand-picked either: it is derived as a TARGET OCCUPANCY from the FIT WINDOW
# ONLY (causal; computed once from bars <= n_fit, never per bar).
#     H_TARGET_RATE = 0.25  -> enter threshold = the (1 - 0.25) quantile of
#                              |trend_t| over the leading n_fit bars
#     H_EXIT_SLACK  = 0.10  -> exit  threshold = the (1 - 0.35) quantile, i.e. a
#                              LOOSER bar, preserving FIX 3's enter/exit band
# The derived thresholds are PRINTED on every run (Section 7.0).
#
# INTENSITY_MODE = 'frozen_z' reproduces today's behaviour BIT-FOR-BIT and is
# asserted to do so in Section 7H-viii against a frozen verbatim copy of the
# pre-change `label_bars`.
INTENSITY_MODE = 'vol_norm'    # 'frozen_z' = pre-change | 'vol_norm' = adopted
MOM_3D_BARS    = BASE_WIN['MOM_3D']    # 9 -- the horizon mom_3d integrates over
H_TARGET_RATE  = 0.25   # target FIT-WINDOW occupancy of the H gate
H_EXIT_SLACK   = 0.10   # exit threshold sits at (H_TARGET_RATE + this) occupancy

# ---------------------------------------------------------------------------
# FIX 6 -- DIRECTION_MODE: how an HMM state maps to bear / side / bull.
#
# 'rank' (DEFAULT, ADOPTED) -- today's hard rank buckets: sort the N states by
# composite bullishness, bottom 2 bear, middle 1 side, top 2 bull. Rank-based so
# the side bucket is guaranteed non-empty (a sign+deadzone rule can empty it and
# silently make SIDEWAYS unreachable -- that bug was shipped once and reverted).
#
# 'soft' (IMPLEMENTED, SWITCHABLE, *NOT* ADOPTED) -- replace the step function of
# the RANK with a smooth function of the VALUE: two sigmoids on the cross-state
# standardized composite score give each state a (bull, side, bear) weight row,
# and the bar's masses become `probs @ W` instead of a hard column sum.
#
# WHY IT IS NOT ADOPTED, stated exactly. Soft bucketing improves stability AT THE
# SOURCE -- the state->direction map stops flipping wholesale when one state's
# score crosses another's, measured 3.2x more stable -- but it makes the EMITTED
# SIDEWAYS/BEAR occupancy gaps WORSE. The reason is the standardization: with
# only N=5 states the composite scores are standardized by the sd of those same 5
# numbers, so a single outlier state inflates the sd and drags every other
# state's z toward zero, which washes the whole map toward SIDEWAYS by a
# different amount in each fit. Until that standardization is fixed (a robust
# scale, or a scale that does not depend on the state count), 'soft' is NOT
# RECOMMENDED and 'rank' remains the default.
DIRECTION_MODE = 'rank'   # 'rank' = adopted | 'soft' = implemented, not recommended
DIR_TAU   = 0.6    # soft: crossover WIDTH of the bull/bear sigmoids (cross-state sd units)
DIR_C     = 0.5    # soft: crossover CENTRE -- how far from the cross-state mean a
                   #       state must sit before it counts as directional
DIR_SCALE = 'sd'   # soft: cross-state scale, 'sd' or 'mad' (robust). This is the
                   #       knob named in the paragraph above; 'mad' is the obvious
                   #       first thing to try when fixing the standardization.

# ---------------------------------------------------------------------------
# ESCALATION_DURING_HOLD -- what the intensity gate may do while the DIRECTION
# is being held by CONFIRM_BARS.
#
# THE COUPLING. FIX 2 holds the emitted DIRECTION for CONFIRM_BARS bars while a
# candidate flip confirms. The intensity gate is graded FRESH at every bar on a
# SEPARATE clock (its own enter/exit state machine). So on the bars where
# dir_raw != dir_emit -- measured at ~5-11% of bars -- the notebook emits a
# direction that the current evidence no longer supports, while the gate is free
# to escalate that stale direction to H. That is maximum conviction emitted at
# maximum uncertainty.
#
#   'allow'            -- the PRE-CHANGE behaviour. Escalation is independent of
#                         whether the direction is contested. Retained as the
#                         bit-for-bit off switch (asserted in 7H-viii).
#   'block'  (DEFAULT) -- no NEW escalation to H while dir_raw != dir_emit. An
#                         already-running H escalation is still held under the
#                         exit band; only fresh ENTERs are suppressed. A blocked
#                         escalation leaves the gate state at 0, so a LATER bar
#                         cannot "hold" an H run it never entered -- the
#                         suppression propagates forward past the contested bar,
#                         which is what keeps the ENTER-band invariant intact.
#   'demote'           -- as 'block', and additionally force an existing H down to
#                         L while contested. The gate run ENDS, so re-escalation
#                         after the contest resolves must clear the full ENTER
#                         band again.
#
# WHY 'block' IS THE DEFAULT -- and, precisely, what is NOT established.
#
# ESTABLISHED (contested-H prototype):
#   * Contested-H underperforms confirmed-H in 18/18 measured cells (2 fit cuts x
#     3 horizons x {bull, bear, pooled}). Every one of the 18 is negative.
#   * MECHANICAL CORROBORATION, independently verified: contested-H bars carry
#     trend_efficiency 0.365 / 0.445 versus 0.597 / 0.603 for confirmed-H -- i.e.
#     contested-H bars are 1.35-1.64x CHOPPIER. That is exactly the condition this
#     system is supposed to take SMALLER size in, and 'allow' prints MAXIMUM size
#     there.
#   * The cost of switching is negligible and structurally safe: only 6-7 bars
#     change (0.39-0.45%), EVERY change is an H -> L demotion, and NO bar changes
#     DIRECTION. 'block' therefore cannot introduce a new failure mode -- it can
#     only reduce conviction.
#   * Persistence slightly IMPROVES under 'block' (runs 293 -> 291), whereas
#     'demote' fragments runs (-> 311). Hence 'block', not 'demote'.
#
# NOT ESTABLISHED -- read this before quoting the above as a return result:
#   * NO single return comparison clears |t| >= 2 under BOTH the HAC and the n_eff
#     corrections. There are only 16-20 contested-H bars in the sample. That is a
#     POWER limitation, not evidence of no effect -- but it means the 18/18 is a
#     consistent DIRECTION, not a demonstrated return gain.
#   * The justification for defaulting this ON is therefore: consistent sign +
#     the efficiency evidence + the asymmetry of costs (a wrongly-suppressed H
#     costs a little upside; a wrongly-emitted H costs full size into chop).
#     It is NOT "contested-H loses money, significantly". Do not overstate it.
#
# CAUSALITY. Both dir_raw and dir_emit are computable from bars <= t (dir_raw is
# a per-bar argmax of causal masses; dir_emit is confirm_delay's left-to-right
# scan), so the contested mask is causal, and the suppression is applied inside
# the same single left-to-right pass the gate already used. This is PROVED by a
# prefix-truncation probe in Section 7.0 under all three settings, not asserted.
ESCALATION_DURING_HOLD = 'block'   # 'block' (adopted) | 'allow' (pre-change) | 'demote'

# Forward-return horizons used for EVALUATION ONLY (hindsight; never a label input).
FWD_HORIZONS = [3, 9, 15]        # ~1 day, ~3 days, ~5 days of 2h bars

# ---------------------------------------------------------------------------
# The three configs under test: V1.0 (prod baseline) vs Candidate A (lean-cov)
# vs Candidate B (lean-feat). All at N=5, CONF_L=0.5. This head-to-head is about
# MODEL CAPACITY (covariance type / feature subset) -- the labeling scheme below
# is identical for all three, so the comparison isolates capacity.
# ---------------------------------------------------------------------------
CONFIGS = [
    dict(name='V1.0 (prod)',  N=5, cov='full', features=FEATURE_COLS),
    dict(name='A: lean-cov',  N=5, cov='diag', features=FEATURE_COLS),
    dict(name='B: lean-feat', N=5, cov='diag', features=FEATURES_LEAN),
]

# ---------------------------------------------------------------------------
# THE ADOPTED CONFIG -- set EXPLICITLY, not inherited from the head-to-head.
#
# The head-to-head (Section 5) still runs in full and still reports its winner.
# But that ranking is KNOWN UNSTABLE: it is decided on worst-fold Sharpe, a
# single noisy number over 4 folds, and a 3-bar perturbation of the input series
# has already been observed to flip it. Letting the whole evaluation suite follow
# whichever config happened to win means the notebook can silently evaluate a
# different detector on two runs of the same code.
#
# So the evaluated config is PINNED here. Sections 5c and 7 both use it, which is
# also what makes those two overlays comparable: they then differ ONLY in the fit
# window (see the re-role note in 5c / 7A), not in model capacity.
#
# If the head-to-head winner differs from this, that DISAGREEMENT IS REPORTED
# loudly in Sections 5b, 7.0 and 8a rather than silently resolved either way.
ADOPTED_CONFIG_NAME = 'A: lean-cov'    # N=5, cov='diag', all 9 features
assert any(c['name'] == ADOPTED_CONFIG_NAME for c in CONFIGS), \
    f'ADOPTED_CONFIG_NAME {ADOPTED_CONFIG_NAME!r} is not one of the configs under test'

# ---- knob sanity (each of these has bitten this project at least once) ------
assert INTENSITY_MODE in ('frozen_z', 'vol_norm'), 'unknown INTENSITY_MODE'
assert DIRECTION_MODE in ('rank', 'soft'), 'unknown DIRECTION_MODE'
assert ESCALATION_DURING_HOLD in ('allow', 'block', 'demote'), 'unknown ESCALATION_DURING_HOLD'
assert MOM_3D_BARS == BASE_WIN['MOM_3D'] == 9, 'mom_3d is expected to integrate 9 bars'
assert 0.0 < H_TARGET_RATE < 1.0 and H_EXIT_SLACK >= 0.0
assert DIR_TAU > 0.0 and DIR_C >= 0.0 and DIR_SCALE in ('sd', 'mad')
assert not (set(BAR_DIR_FEATURES) & {'vol_2h', 'vol_expansion'}), \
    'a MAGNITUDE feature leaked into the per-bar DIRECTION score'

print(f"Labeling: direction+intensity  CONF_L={CONF_L}  Z_HI={Z_HI}  "
      f"EFF_HI={EFF_HI}  EFF_WIN={EFF_WIN}  TREND_FEATURE={TREND_FEATURE}")
print(f"Fixes   : [1] DIRECTION_EXCLUDE={DIRECTION_EXCLUDE or '() -> old scorer'}  "
      f"[2] CONFIRM_BARS={CONFIRM_BARS}"
      + ('  (=1 -> old behaviour)' if CONFIRM_BARS == 1 else '')
      + f"  [3] gate bands enter(|z|>={Z_HI}, eff>={EFF_HI}) "
        f"exit(|z|<{Z_HI_EXIT}, eff<{EFF_HI_EXIT})"
      + ('  (exit==enter -> old behaviour)'
         if (Z_HI_EXIT == Z_HI and EFF_HI_EXIT == EFF_HI) else ''))
print(f"          [4] BAR_DIR_WEIGHT={BAR_DIR_WEIGHT} (per-bar direction blend; "
      f"tau={BAR_DIR_TAU}, features={list(BAR_DIR_FEATURES)})")
if BAR_DIR_WEIGHT == 0.0:
    print("              FIX 4 IS RETAINED AS A SWITCH BUT SET OFF. Direction is "
          "100% per-STATE (HMM).")
    print("              WHY: it broke the direction-level forward-return ordering, "
          "and the rally")
    print("              benefit it was added for did not reproduce across runs "
          "(+19.7pp, then +1.4pp,")
    print("              then a run where the gain came from fixes 1-2 with fix 4 "
          "already off).")
    print("              NOT deleted: the blend, the 7H-vi sweep and the 7H-vii "
          "overlay all still run,")
    print("              and bar_dir_score is still computed (diagnostic only -- the "
          "blend is an exact")
    print("              arithmetic no-op at w=0.0, asserted bit-for-bit in 7H).")
else:
    print(f"              FIX 4 IS ON. NOTE: w={BAR_DIR_WEIGHT} is NOT the shipped "
          "default (0.0); it broke")
    print("              the direction-level forward-return ordering on real data.")
print(f"Fitting : SEED ENSEMBLE  ENSEMBLE_K={ENSEMBLE_K}  BASE_SEED={BASE_SEED}  "
      f"seeds={[BASE_SEED + i for i in range(ENSEMBLE_K)]}"
      + ('   (K=1 -> old single-fit behaviour)' if ENSEMBLE_K == 1 else ''))
print(f"          [5] INTENSITY_MODE={INTENSITY_MODE!r}"
      + ("  (scale-free t-stat gate; thresholds derived from the FIT WINDOW at "
         f"target occupancy {H_TARGET_RATE:.2f} enter / {H_TARGET_RATE + H_EXIT_SLACK:.2f} exit)"
         if INTENSITY_MODE == 'vol_norm'
         else f"  (pre-change frozen mu/sd z-score vs constants Z_HI={Z_HI}/Z_HI_EXIT={Z_HI_EXIT})"))
print(f"          [6] DIRECTION_MODE={DIRECTION_MODE!r}"
      + ("  (hard rank buckets -- adopted)" if DIRECTION_MODE == 'rank'
         else f"  (SOFT weights tau={DIR_TAU} c={DIR_C} scale={DIR_SCALE!r}"
              "  -- NOT RECOMMENDED, see the note in the config cell)"))
print(f"          [7] ESCALATION_DURING_HOLD={ESCALATION_DURING_HOLD!r}"
      + ("  (=allow -> PRE-CHANGE behaviour: the gate may escalate a held/contested direction)"
         if ESCALATION_DURING_HOLD == 'allow'
         else "  (no NEW H escalation while dir_raw != dir_emit"
              + ("; existing H also demoted)" if ESCALATION_DURING_HOLD == 'demote' else ")")))
print(f"Harness : HMM_ITER={HMM_ITER}  N_FOLDS={N_FOLDS}  MIN_TRAIN_FRAC={MIN_TRAIN_FRAC}  "
      f"configs={[c['name'] for c in CONFIGS]}")
print(f"Adopted : evaluated config is PINNED to {ADOPTED_CONFIG_NAME!r} (Sections 5c and 7); "
      f"the head-to-head still runs and its winner is reported and compared in 5b / 7.0 / 8a")
print(f"Runtime : N_JOBS={N_JOBS} (ensemble fits" + ('  serial' if N_JOBS == 1 else ' in parallel')
      + f")  RUN_SEED_STABILITY={RUN_SEED_STABILITY}"
      + ('  (5d diagnostic SKIPPED -- see 5d)' if not RUN_SEED_STABILITY else '')
      + '   [speed only; neither knob can change a result]')

In [ ]:
# ==========================================================================
# ENGINE CELL 2/4 -- synthetic 2h generator (VERBATIM; the yfinance
# driver line is dropped, yfinance is firewalled in this environment)
# ==========================================================================
def load_2h():
    """yfinance 60m->2h, else synthetic. Returns (nifty, vix, is_synth)."""
    try:
        import yfinance as yf

        def h2(tk):
            raw = yf.download(tk, interval='60m', period='730d',
                              auto_adjust=True, progress=False)
            if raw is None or len(raw) == 0:
                return None
            c = raw['Close'].squeeze().dropna()
            idx = pd.to_datetime(c.index)
            c.index = idx.tz_localize(None) if idx.tz is not None else idx   # naive index
            return c.resample('2h').last().dropna()

        nifty = h2('^NSEI')
        if nifty is not None and len(nifty) > 200:
            vix = h2('^INDIAVIX')
            vix = (vix.reindex(nifty.index).ffill().bfill()
                   if vix is not None and len(vix) else pd.Series(15.0, index=nifty.index))
            nifty.name, vix.name = 'nifty', 'vix'
            print(f'yfinance 60m->2h: {len(nifty)} bars')
            return nifty, vix, False
        raise ValueError('too few bars')
    except Exception as e:
        print(f'yfinance unavailable ({e}); falling back to synthetic 2h data.')
        return _synth()


def _synth():
    """Regime-blocked GBM + OU VIX at 2h cadence (offline validation only)."""
    rng = np.random.default_rng(42)
    days = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=520)
    idx = pd.DatetimeIndex([d + pd.Timedelta(hours=h) for d in days for h in (10, 12, 14)])
    REG = [(0.00, 0.15,  0.0004, 0.0035, 14.0, 0.6),
           (0.15, 0.30, -0.0006, 0.0060, 22.0, 1.1),
           (0.30, 0.45,  0.0001, 0.0030, 16.0, 0.7),
           (0.45, 0.55, -0.0030, 0.0110, 45.0, 2.5),
           (0.55, 0.75,  0.0006, 0.0050, 20.0, 1.0),
           (0.75, 0.90,  0.0005, 0.0032, 13.5, 0.6),
           (0.90, 1.00, -0.0002, 0.0045, 18.0, 0.9)]
    n, lvl, pv, nv, vv = len(idx), 18000.0, 15.0, [], []
    for i in range(n):
        f = i / n
        mu, sg, vm, vf = 0.0002, 0.0040, 16.0, 0.8
        for fs, fe, m, s, v, q in REG:
            if fs <= f < fe:
                mu, sg, vm, vf = m, s, v, q
                break
        lvl *= np.exp(rng.normal(mu, sg))
        pv = max(pv + 0.05 * (vm - pv) + rng.normal(0, vm * 0.08 * vf), 8.0)
        nv.append(lvl); vv.append(pv)
    return (pd.Series(nv, index=idx, name='nifty'),
            pd.Series(vv, index=idx, name='vix'), True)

In [ ]:
# ==========================================================================
# ENGINE CELL 3/4 -- feature + labeling engine (VERBATIM)
# ==========================================================================
def build_features(nifty, vix, scale=LOOKBACK_SCALE):
    """The 9 causal swing features (engine column names)."""
    w = {k: max(2, int(round(v * scale))) for k, v in BASE_WIN.items()}
    r = np.log(nifty / nifty.shift(1))
    df = pd.DataFrame(index=nifty.index)
    df['ret_2h'] = r
    df['mom_1d'] = r.rolling(w['MOM_1D']).sum()
    df['mom_3d'] = r.rolling(w['MOM_3D']).sum()
    df['mom_5d'] = r.rolling(w['MOM_5D']).sum()
    df['vol_2h'] = r.rolling(w['VOL_WIN']).std()
    df['vol_expansion'] = r.rolling(w['VOL_FAST']).std() / r.rolling(w['VOL_SLOW']).std()
    df['vix_chg'] = (vix - vix.shift(1)) / vix.shift(1)
    sh = nifty.rolling(w['SWING_WIN']).max()
    df['drawdown'] = (sh - nifty) / sh
    ma = nifty.rolling(w['SWING_WIN']).mean()
    df['dist_ma'] = (nifty - ma) / ma
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    return df


def direction_weight(feat, exclude=None):
    """Weight this feature contributes to the composite BULLISHNESS score.

    FIX 1: features listed in `exclude` (default DIRECTION_EXCLUDE) get weight
    0.0 -- they stop voting on DIRECTION while remaining full HMM inputs.
    vol_2h and vol_expansion are magnitude measures, not signed ones; scoring a
    high-volatility RALLY as bearish was the bug this removes.

    Subset-agnostic: excluding a feature that is not in the active subset is a
    no-op, and `exclude=()` reproduces the original weights exactly.
    """
    exclude = DIRECTION_EXCLUDE if exclude is None else exclude
    if feat in exclude:
        return 0.0
    return FEATURE_SIGN[feat] * FEATURE_MAG.get(feat, 1.0)


def composite_subset(means, feature_subset, exclude=None):
    """Bullishness score per state; works with ANY feature subset."""
    score = np.zeros(means.shape[0])
    for j, feat in enumerate(feature_subset):
        score += direction_weight(feat, exclude) * means[:, j]
    # A subset whose every feature is excluded would make all states score 0 and
    # the direction ranking arbitrary -- catch that rather than emit noise.
    assert any(direction_weight(f, exclude) != 0.0 for f in feature_subset), \
        'DIRECTION_EXCLUDE removed every feature from the direction score'
    return score


def n_params(N, F, covariance_type):
    """GaussianHMM free parameters: means + covariances + transitions + startprob."""
    cov = N * F * (F + 1) / 2 if covariance_type == 'full' else N * F
    return N * F + cov + N * (N - 1) + (N - 1)


def trend_efficiency(close, win=EFF_WIN):
    """Causal Kaufman efficiency ratio over `win` bars:

        |close[t] - close[t-win]| / sum(|close.diff()|)

    i.e. net displacement / total path walked. ~1.0 in a straight-line trend,
    ~0.0 when price keeps doubling back inside a range -- the distinction raw
    momentum magnitude cannot make. Uses only bars <= t.
    """
    net = (close - close.shift(win)).abs()
    path = close.diff().abs().rolling(win).sum()
    return (net / path.replace(0, np.nan)).fillna(0.0).clip(0.0, 1.0)


def gate_band(trend_raw, feat, n_fit, mode=None, thr_enter=None, thr_exit=None,
              target_rate=None, exit_slack=None, hysteresis=None):
    """ONE band contract for BOTH intensity modes.

    Returns `(trend, thr_enter, thr_exit)`: the trend-MAGNITUDE series and the
    (enter, exit) thresholds it is graded against.

    THIS FUNCTION EXISTS TO RESOLVE A REAL COUPLING. In the prototype, the gate
    BAND (FIX 3's Z_HI_EXIT / EFF_HI_EXIT) and the INTENSITY MODE (FIX 5) were
    mutually exclusive: the vol_norm branch asserted `z_exit is None`, because
    z_exit was a threshold expressed in frozen-z units and vol_norm derives its
    thresholds by occupancy instead. That is a units problem, not a logic
    problem, and it is fixed here by making the BAND -- not the threshold
    constants -- the shared abstraction. Both modes now:

      * produce a trend series in their OWN units,
      * carry a DEFAULT (enter, exit) band in those same units,
      * accept an OCCUPANCY-derived band (`target_rate` / `exit_slack`) computed
        on the FIT WINDOW of that same series,
      * accept EXPLICIT overrides (`thr_enter` / `thr_exit`) in those same units,
      * accept `hysteresis=False`, a MODE-INDEPENDENT way to say "no band"
        (exit == enter), which is what FIX-3-off means in either mode.

    So the mode and the band are now independent knobs, and nothing is papered
    over with an assert.

    mode='frozen_z' : trend = (mom_3d - mu_fit)/sd_fit.
                      Default band = (Z_HI, Z_HI_EXIT), the pre-change constants.
    mode='vol_norm' : trend = mom_3d / (vol_2h * sqrt(MOM_3D_BARS)) -- the
                      t-statistic of the 9-bar move: dimensionless,
                      contemporaneous, needing NO fit-window baseline.
                      Default band = occupancy quantiles at H_TARGET_RATE /
                      H_TARGET_RATE + H_EXIT_SLACK.

    PRECEDENCE (most specific wins): explicit thr_* > occupancy (target_rate /
    exit_slack, honoured under EITHER mode) > the mode default. `hysteresis=False`
    is applied last and collapses exit onto enter whatever produced them.

    CAUSALITY. Under frozen_z, mu/sd come from the leading n_fit bars. Under
    vol_norm the SERIES needs no fit window at all (mom_3d and vol_2h are both
    trailing rolling windows at bar t) and the THRESHOLDS are quantiles over the
    leading n_fit bars only -- computed ONCE, never per bar, never over bars the
    model has not seen. Either way a prefix truncation at any t >= n_fit
    reproduces both the series value at t and the thresholds exactly. Section 7.0
    proves this by truncation rather than asserting it here.
    """
    mode = INTENSITY_MODE if mode is None else mode
    assert mode in ('frozen_z', 'vol_norm'), f'unknown INTENSITY_MODE {mode!r}'
    tr = np.asarray(trend_raw, dtype=float)
    n_fit = int(n_fit)
    assert 2 <= n_fit <= len(tr), 'fit window out of range for the intensity gate'

    if mode == 'frozen_z':
        mu = float(np.mean(tr[:n_fit]))
        sd = float(np.std(tr[:n_fit]))
        trend = (tr - mu) / (sd if sd > 0 else 1.0)
        ok = np.isfinite(trend)
        d_enter, d_exit = float(Z_HI), float(Z_HI_EXIT)
    else:
        assert feat is not None and 'vol_2h' in getattr(feat, 'columns', []), \
            "vol_norm needs the raw feature frame (for vol_2h)"
        vol = np.asarray(feat['vol_2h'].values, dtype=float)
        assert len(vol) == len(tr), 'vol_2h must be aligned 1:1 with the trend feature'
        # DENOMINATOR GUARD: vol_2h is NaN through warm-up and 0.0 on a dead-flat
        # stretch. Those bars get trend = 0.0 -- the neutral value, which fires no
        # gate and can only SUPPRESS an escalation, never invent one.
        den = vol * np.sqrt(MOM_3D_BARS)
        ok = np.isfinite(den) & (den > 0.0) & np.isfinite(tr)
        trend = np.zeros(len(tr), dtype=float)
        np.divide(tr, den, out=trend, where=ok)
        trend[~np.isfinite(trend)] = 0.0
        d_enter = d_exit = None                 # derived by occupancy below

    # ---- occupancy-derived band (either mode) -----------------------------
    if d_enter is None or target_rate is not None or exit_slack is not None:
        tgt = H_TARGET_RATE if target_rate is None else float(target_rate)
        slk = H_EXIT_SLACK if exit_slack is None else float(exit_slack)
        assert 0.0 < tgt < 1.0 and slk >= 0.0, 'target occupancy out of range'
        a = np.abs(trend[:n_fit])[ok[:n_fit]]   # guarded bars excluded from the quantile
        assert a.size >= 20, 'too few usable fit-window bars to derive a threshold'
        d_enter = float(np.quantile(a, 1.0 - tgt))
        d_exit = float(np.quantile(a, 1.0 - min(tgt + slk, 0.999)))

    en = float(d_enter) if thr_enter is None else float(thr_enter)
    ex = float(d_exit) if thr_exit is None else float(thr_exit)
    if hysteresis is False:
        ex = en                                 # FIX-3-OFF, stated mode-independently
    ex = min(ex, en)                            # the exit band must be the looser one
    return trend, en, ex


def intensity_state(z, eff, z_hi=None, eff_hi=None, z_exit=None, eff_exit=None,
                    block_enter=None, force_exit=None):
    """FIX 3 -- gate hysteresis. Returns a signed intensity array:

        +1  bar is escalated H on the BULL side
        -1  bar is escalated H on the BEAR side
         0  bar stays L

    Enter (0 -> +/-1): |z| >= z_hi AND eff >= eff_hi          (unchanged gates)
    Hold  (stay +/-1): |z| >= z_exit AND eff >= eff_exit AND sign(z) unchanged
    Exit  (-> 0):      |z| <  z_exit OR  eff <  eff_exit OR  sign(z) flipped

    A sign flip forces a fresh ENTER test rather than silently relabelling a held
    bull escalation as a bear one.

    CAUSAL: a single left-to-right scan whose state at bar t depends only on
    bars <= t, so truncating the input after t cannot change out[t].

    z_exit == z_hi and eff_exit == eff_hi reduce this EXACTLY to the old
    memoryless `(|z| >= z_hi) & (eff >= eff_hi)` test.

    ESCALATION_DURING_HOLD support -- two OPTIONAL per-bar boolean masks, applied
    INSIDE this same single pass so the state machine stays consistent (a
    suppressed escalation must not be silently "held" on a later bar as though it
    had happened):

      block_enter[t] : the ENTER test is skipped at bar t. An already-running
                       escalation is unaffected and is still held under the exit
                       band. This is 'block'.
      force_exit[t]  : the state is additionally forced to 0 at bar t, ending the
                       run. Re-escalation later must clear the full ENTER band
                       again. This is the extra half of 'demote'.

    Both masks default to all-False, in which case every branch below is exactly
    the pre-change scan -- and Section 7H-viii asserts that bit-for-bit rather
    than trusting this paragraph. Neither mask can CREATE an escalation; both can
    only suppress one, so no chop-filter invariant can be weakened by them.

    CAUSALITY IS PRESERVED BY CONSTRUCTION: mask[t] is consumed at step t of a
    left-to-right scan, so out[t] still depends only on (z, eff, masks)[0..t].
    """
    z_hi = Z_HI if z_hi is None else z_hi
    eff_hi = EFF_HI if eff_hi is None else eff_hi
    z_exit = Z_HI_EXIT if z_exit is None else z_exit
    eff_exit = EFF_HI_EXIT if eff_exit is None else eff_exit
    z = np.asarray(z, dtype=float)
    eff = np.asarray(eff, dtype=float)
    n = len(z)
    _blk = (np.zeros(n, dtype=bool) if block_enter is None
            else np.asarray(block_enter, dtype=bool))
    _fex = (np.zeros(n, dtype=bool) if force_exit is None
            else np.asarray(force_exit, dtype=bool))
    assert len(_blk) == n and len(_fex) == n, 'gate suppression masks must align 1:1 with z'
    out = np.zeros(n, dtype=int)
    state = 0
    for t in range(n):
        s = 1 if z[t] > 0 else (-1 if z[t] < 0 else 0)
        if state != 0 and s == state:
            if abs(z[t]) < z_exit or eff[t] < eff_exit:
                state = 0                       # de-escalate on the EXIT band
        else:
            state = 0                           # no state, or the sign flipped
        if _fex[t]:
            state = 0                           # 'demote': drop a held H to L
        if (state == 0 and not _blk[t]
                and abs(z[t]) >= z_hi and eff[t] >= eff_hi):
            state = s                           # escalate on the ENTER band
        out[t] = state
    return out


def hold_masks(dir_raw, dir_emit, policy=None):
    """(block_enter, force_exit) for ESCALATION_DURING_HOLD.

    A bar is CONTESTED when the emitted direction is not the direction the
    current bar's own evidence votes for -- i.e. `dir_raw[t] != dir_emit[t]`,
    which is exactly the set of bars CONFIRM_BARS is holding through. Both inputs
    are computable from bars <= t, so the mask is causal.

    'allow'  -> (all False, all False)  == unchanged behaviour, bit-for-bit.
    'block'  -> (contested, all False)  == no NEW escalation while contested.
    'demote' -> (contested, contested)  == also drop an existing H to L.
    """
    policy = ESCALATION_DURING_HOLD if policy is None else policy
    assert policy in ('allow', 'block', 'demote'), f'unknown ESCALATION_DURING_HOLD {policy!r}'
    n = len(dir_raw)
    contested = np.asarray(dir_raw) != np.asarray(dir_emit)
    if policy == 'allow':
        z = np.zeros(n, dtype=bool)
        return z, z.copy(), contested
    if policy == 'block':
        return contested, np.zeros(n, dtype=bool), contested
    return contested, contested.copy(), contested


def confirm_delay(raw, confirm_bars=None):
    """FIX 2 -- causal label hysteresis (a CONFIRMATION DELAY, not a smoother).

    A candidate value must be observed on `confirm_bars` CONSECUTIVE bars before
    the emitted series is allowed to flip to it; until then the previous emitted
    value is held.

    WHY THIS IS CAUSAL, stated precisely: out[t] is a function of raw[0..t] only.
    The scan never looks at raw[t+1..]. Contrast with the look-ahead smoother
    this project previously removed, which decided whether to erase a run by
    inspecting that run's full REALIZED length -- i.e. it needed bars after t to
    decide bar t. This does the opposite: it PAYS a delay rather than borrowing
    the future. A flip that turns out to be a 1-bar blip is simply never emitted;
    a flip that persists is emitted `confirm_bars - 1` bars late.

    confirm_bars = 1 reproduces the input exactly (out is raw).
    """
    confirm_bars = CONFIRM_BARS if confirm_bars is None else int(confirm_bars)
    assert confirm_bars >= 1, 'CONFIRM_BARS must be >= 1'
    raw = np.asarray(raw)
    n = len(raw)
    out = np.empty(n, dtype=raw.dtype)
    if n == 0:
        return out
    out[0] = raw[0]                 # bar 0 has no prior emitted label to hold
    cand, run = raw[0], 0
    for t in range(1, n):
        if raw[t] == out[t - 1]:
            out[t] = raw[t]         # agrees with what is already emitted
            cand, run = raw[t], 0
        else:
            if raw[t] == cand:
                run += 1
            else:
                cand, run = raw[t], 1
            if run >= confirm_bars:
                out[t] = cand       # candidate confirmed -> flip
                run = 0
            else:
                out[t] = out[t - 1]  # not yet confirmed -> HOLD the old label
    return out


def direction_buckets(means, feature_subset, exclude=None):
    """Bucket HMM states into bear(-1) / side(0) / bull(+1) by RANK of composite
    bullishness. Rank-based (not a sign+deadzone threshold) so the side bucket is
    guaranteed non-empty; a deadzone can degenerate to an empty side bucket, which
    silently makes SIDEWAYS unreachable. For N=5 this is: bottom 2 bear, middle 1
    side, top 2 bull.

    `exclude` is threaded to the scorer (FIX 1); None uses DIRECTION_EXCLUDE.
    """
    scores = composite_subset(means, feature_subset, exclude)
    n_st = len(scores)
    order = np.argsort(scores)                 # ascending bullishness
    n_side = max(1, round(n_st / 5))
    n_bear = (n_st - n_side) // 2
    direction = np.empty(n_st, dtype=int)
    direction[order[:n_bear]] = -1
    direction[order[n_bear:n_bear + n_side]] = 0
    direction[order[n_bear + n_side:]] = 1
    # INVARIANT: every direction bucket must be reachable, else whole labels vanish.
    assert (direction == 1).sum() >= 1, 'bull bucket empty -> H_BULL/L_BULL unreachable'
    assert (direction == -1).sum() >= 1, 'bear bucket empty -> H_BEAR/L_BEAR unreachable'
    assert (direction == 0).sum() >= 1, 'side bucket empty -> SIDEWAYS unreachable'
    return direction


def bar_direction_score(dir_feats, n_fit, features=None):
    """FIX 4 -- CAUSAL PER-BAR directional z-score.

    dir_feats : DataFrame of RAW (unscaled) features aligned to the labeled bars,
                one row per bar, in bar order. Only the signed directional columns
                are read.
    n_fit     : number of LEADING bars that constitute the fit window. The mu/sd
                baseline is computed from those rows ONLY -- exactly the way
                trend_z's baseline is frozen -- so no bar > t and no bar the model
                has not seen can influence bar t's score.

    Construction:
      1. keep only BAR_DIR_FEATURES that are present (ret_2h, mom_1d, mom_3d,
         mom_5d, dist_ma -- all SIGNED directional measures);
      2. z-score each against its FIT-WINDOW mu/sd;
      3. weighted mean with the existing FEATURE_SIGN * FEATURE_MAG weights,
         normalised by sum|w| so the composite stays on a z-like scale;
      4. re-standardize the composite against ITS fit-window mu/sd, so the output
         is a unit-variance z on the fit window and BAR_DIR_TAU is interpretable.

    vol_2h / vol_expansion are structurally barred: they measure how BIG a move
    is, not which way it points, and letting a magnitude term vote on direction
    is the category error this whole fix exists to undo. Asserted below.

    Causality: every input column is a backward-looking rolling statistic, and
    the baseline uses leading rows only, so score[t] depends on bars <= t alone.
    Recomputing from the prefix dir_feats.iloc[:t+1] reproduces score[t] exactly
    (probed in 7.0).
    """
    features = BAR_DIR_FEATURES if features is None else tuple(features)
    assert not (set(features) & {'vol_2h', 'vol_expansion'}), \
        'per-bar DIRECTION score must not contain magnitude features'
    cols = [f for f in features if f in dir_feats.columns]
    assert cols, 'no directional features available for the per-bar direction score'
    A = np.asarray(dir_feats[cols].values, dtype=float)
    assert n_fit >= 2 and n_fit <= len(A), 'fit window out of range for the per-bar score'
    mu = A[:n_fit].mean(axis=0)
    sd = A[:n_fit].std(axis=0)
    Z = (A - mu) / np.where(sd > 0, sd, 1.0)
    # exclude=() deliberately: DIRECTION_EXCLUDE is FIX 1's state-level knob and
    # must not be able to mute a feature that is already guaranteed directional.
    w = np.array([direction_weight(f, exclude=()) for f in cols], dtype=float)
    assert np.abs(w).sum() > 0, 'per-bar direction weights are all zero'
    # (Z * w).sum(axis=1), NOT Z @ w. The two are algebraically identical and the
    # matmul is the obvious way to write it -- but `DataFrame.values` hands back an
    # F-ORDERED array, and BLAS gemv on an F-ordered operand picks its blocking
    # from the ROW COUNT, so the last-bar result changes in the last ulp depending
    # on how many bars follow it. That is a ~1e-16 difference with no economic
    # meaning, but it makes the truncation probe in 7.0 fail its bit-exactness
    # test, and a causality probe that has to be run at a tolerance is a weaker
    # probe. The row-wise form sums 5 terms per row independently of the array
    # length, so score[t] is BIT-identical whether or not bars > t exist -- and
    # 7.0 can therefore assert exact equality rather than np.isclose.
    s = (Z * w).sum(axis=1) / np.abs(w).sum()
    s_mu = float(s[:n_fit].mean())
    s_sd = float(s[:n_fit].std())
    return (s - s_mu) / (s_sd if s_sd > 0 else 1.0)


def bar_direction_masses(score, tau=None):
    """Map the per-bar directional z to bull / side / bear masses that sum to 1.

    Softmax over the three logits (+s/tau, 0, -s/tau): bull dominates for s >> 0,
    bear for s << 0, and SIDE is the plurality only near s ~ 0 -- which is the
    right shape, because "no clear direction at this bar" is a real answer and
    must remain reachable. Computed in a shift-stabilised form so large |s| does
    not overflow.

    Returns (bull, side, bear), each an array over bars, summing to 1 per bar.
    """
    tau = BAR_DIR_TAU if tau is None else float(tau)
    e = np.asarray(score, dtype=float) / max(tau, 1e-12)
    a = np.abs(e)                                   # = max(e, 0, -e), the shift
    eb, es, er = np.exp(e - a), np.exp(-a), np.exp(-e - a)
    tot = eb + es + er
    bull, side, bear = eb / tot, es / tot, er / tot
    assert np.allclose(bull + side + bear, 1.0, atol=1e-9), \
        'per-bar direction masses must partition to 1'
    return bull, side, bear


def _filtered_posteriors(model, X):
    """
    CAUSAL (filtered) state posteriors: P(state_t | observations_1..t).

    Why this exists instead of model.predict_proba():
      hmmlearn's predict_proba runs forward-BACKWARD, so the posterior it
      reports for bar t is smoothed using the whole sequence — including bars
      AFTER t. That is legitimate for offline sequence analysis but is
      look-ahead for a trading regime label: on 2h Nifty data it changes the
      winning state on ~6% of bars versus what was actually knowable at the
      time. predict() (Viterbi) has the same whole-sequence property.

      This is the forward (alpha) recursion only, so each bar's posterior is
      conditioned solely on information available at that bar — exactly what a
      live engine would have. The last bar of a forward-backward pass happens
      to equal the filtered value (no future exists yet), which is why LIVE
      calls were always correct; it is the HISTORICAL labels, and therefore
      every backtest built on them, that needed this fix.

    Computed by the SCALED forward algorithm: alpha is held in LINEAR space and
    renormalised to sum 1 at every step. See `_filtered_posteriors_logspace`
    below for the original log-space/logsumexp formulation, which this is checked
    against bar-by-bar in Section 7.0.

    WHY THE SCALED FORM IS THE SAME ANSWER. The quantity wanted here is the
    NORMALISED filtered posterior at each t, which is scale-free in alpha: for
    any c_t > 0, normalising c_t * alpha_t gives the identical row. So the
    per-step renormalisation -- which is what the log-space version was already
    doing, just via logsumexp -- is not an approximation, it IS the answer. The
    emission frame is likewise exponentiated after subtracting its per-row max,
    another positive per-row constant that cancels in the same normalisation.
    Nothing accumulates, so nothing underflows: alpha sums to 1 after every bar.

    WHY IT IS FASTER. The recursion over t is inherently sequential and is NOT
    vectorised across t (doing so would be wrong). What changes is the cost of
    each step: two scipy.special.logsumexp calls plus an (N,N) broadcast add and
    an exp become one length-N matrix-vector product, one multiply and one
    divide. logsumexp is a Python-level function doing max/subtract/exp/sum/log
    over an (N,N) array per bar; on ~2.9k bars per model per labeling call, and
    hundreds of such calls, that dominates the labeling cost.

    A guard falls back to the log-space implementation if the linear recursion
    ever produces a non-finite or non-positive normaliser, so the fast path can
    never silently return a degraded answer.
    """
    log_frame = model._compute_log_likelihood(X)          # (T, n_states)
    tiny      = np.finfo(float).tiny
    start     = model.startprob_ + tiny
    trans     = model.transmat_ + tiny

    T, N = log_frame.shape
    out  = np.empty((T, N), dtype=float)

    # exp of the emission log-likelihoods, per-bar max removed. The removed
    # factor is a positive per-row constant and cancels in the normalisation.
    frame = np.exp(log_frame - log_frame.max(axis=1, keepdims=True))

    alpha = start * frame[0]
    s = alpha.sum()
    if not (s > 0.0 and np.isfinite(s)):
        return _filtered_posteriors_logspace(model, X)
    alpha = alpha / s
    out[0] = alpha

    for t in range(1, T):
        # predict step (transition) then update step (emission at bar t)
        alpha = (alpha @ trans) * frame[t]
        s = alpha.sum()
        if not (s > 0.0 and np.isfinite(s)):
            return _filtered_posteriors_logspace(model, X)
        alpha = alpha / s                                 # renormalise each step
        out[t] = alpha

    return out


def _filtered_posteriors_logspace(model, X):
    """The ORIGINAL log-space / logsumexp forward recursion.

    Kept verbatim as (a) the reference implementation that
    `_filtered_posteriors` is asserted equal to in Section 7.0, and (b) the
    fallback if the scaled recursion ever hits a degenerate normaliser. Slower,
    but identical in what it computes.
    """
    from scipy.special import logsumexp

    log_frame = model._compute_log_likelihood(X)          # (T, n_states)
    tiny      = np.finfo(float).tiny
    log_start = np.log(model.startprob_ + tiny)
    log_trans = np.log(model.transmat_ + tiny)

    T, N = log_frame.shape
    out  = np.empty((T, N), dtype=float)

    log_alpha = log_start + log_frame[0]
    log_alpha -= logsumexp(log_alpha)
    out[0] = np.exp(log_alpha)

    for t in range(1, T):
        # predict step (transition) then update step (emission at bar t)
        log_alpha = logsumexp(log_alpha[:, None] + log_trans, axis=0) + log_frame[t]
        log_alpha -= logsumexp(log_alpha)                 # renormalise each step
        out[t] = np.exp(log_alpha)

    return out


def ensemble_seeds(K=None, base_seed=None):
    """Deterministic seed list for the ensemble: base_seed + 0..K-1."""
    K = ENSEMBLE_K if K is None else int(K)
    base_seed = BASE_SEED if base_seed is None else int(base_seed)
    return [base_seed + i for i in range(K)]


def _fit_one_hmm(X_train, n_components, covariance_type, n_iter, sd):
    """ONE ensemble member. Top-level (not a closure) so joblib can pickle it.

    Fully determined by its arguments: the seed is explicit, so this touches no
    global RNG state and is identical whether it runs in this process or a
    worker. This is the only place a GaussianHMM is constructed and fit.
    """
    m = hmm.GaussianHMM(n_components=n_components, covariance_type=covariance_type,
                        n_iter=n_iter, random_state=sd,
                        init_params='stmc', params='stmc')
    m.fit(X_train)
    return m


def fit_hmm_ensemble(X_train, n_components, covariance_type,
                     K=None, base_seed=None, n_iter=None):
    """Fit K GaussianHMMs on the SAME training slice with K different seeds.

    This is the identifiability fix. EM is a local optimizer; one seed gives one
    arbitrary local optimum. K seeds give K samples of the optimum set, whose
    direction-bucket masses are averaged in `ensemble_direction_masses` below.

    Every model sees EXACTLY the same rows (`X_train`), which must already be
    the causal leading slice -- this function does no slicing of its own, so it
    cannot introduce look-ahead.

    The K fits are INDEPENDENT and each is fully determined by its own
    random_state, so with N_JOBS != 1 they are dispatched concurrently via
    joblib. That is a pure scheduling change: no fit can observe another, and
    none of them consumes global RNG state (each gets an explicit seed). N_JOBS=1
    takes the plain serial loop. Section 7.0 asserts the two paths return
    BIT-IDENTICAL models.

    Returns (models, all_converged).
    """
    n_iter = HMM_ITER if n_iter is None else int(n_iter)
    seeds = ensemble_seeds(K, base_seed)

    if N_JOBS == 1 or len(seeds) == 1:
        models = [_fit_one_hmm(X_train, n_components, covariance_type, n_iter, sd)
                  for sd in seeds]
    else:
        from joblib import Parallel, delayed
        models = Parallel(n_jobs=N_JOBS, backend='loky')(
            delayed(_fit_one_hmm)(X_train, n_components, covariance_type, n_iter, sd)
            for sd in seeds)

    all_conv = all(bool(m.monitor_.converged) for m in models)
    return list(models), all_conv


def ensemble_direction_masses(models, Xs, feature_subset, exclude=None):
    """Average the direction-bucket probability masses across an ensemble.

    For each fitted model:
      1. CAUSAL filtered posteriors via `_filtered_posteriors` (forward-only
         alpha recursion). Never predict/predict_proba over the whole sequence --
         those are forward-BACKWARD/Viterbi and smooth bar t with bars after t.
      2. `direction_buckets` maps that model's states to bear/side/bull.
      3. The per-state posterior collapses to 3 columns: bull / side / bear.

    Those 3-column arrays are then averaged across models. This is only valid
    because direction masses are PERMUTATION-INVARIANT: model A's "state 3" and
    model B's "state 1" are unrelated integers, but "the probability mass sitting
    in bullish states" means the same thing in both. Averaging raw per-state
    posteriors would be meaningless.

    Causality is preserved exactly: the average of K quantities each of which
    depends only on bars <= t depends only on bars <= t.

    Returns (bull_mass, side_mass, bear_mass, probs_model0), where probs_model0
    is the first model's per-state filtered posterior (used only for the
    `hmm_state_int` reporting column, which has no ensemble analogue).
    """
    models = list(models)
    assert len(models) >= 1, 'ensemble must contain at least one model'
    acc = np.zeros((len(Xs), 3), dtype=float)      # columns: bull, side, bear
    probs0 = None
    for i, m in enumerate(models):
        direction = direction_buckets(m.means_, feature_subset, exclude)
        probs = _filtered_posteriors(m, Xs)        # CAUSAL, forward-only
        if i == 0:
            probs0 = probs
        acc[:, 0] += probs[:, direction == 1].sum(axis=1)
        acc[:, 1] += probs[:, direction == 0].sum(axis=1)
        acc[:, 2] += probs[:, direction == -1].sum(axis=1)
    acc /= len(models)
    # INVARIANT: an average of rows that each sum to 1 must itself sum to 1.
    assert np.allclose(acc.sum(axis=1), 1.0, atol=1e-9), \
        'ensembled direction masses must partition to 1'
    return acc[:, 0], acc[:, 1], acc[:, 2], probs0


# ---------------------------------------------------------------------------
# FIX 6 -- DIRECTION_MODE = 'soft'. IMPLEMENTED AND SWITCHABLE, NOT ADOPTED.
#
# READ THIS BEFORE TURNING IT ON. Soft bucketing improves stability AT THE SOURCE
# (the state -> direction map stops flipping wholesale when one state's composite
# score crosses another's -- measured 3.2x more stable) but it makes the EMITTED
# SIDEWAYS / BEAR occupancy gaps WORSE. The mechanism is the standardization
# below: with only N = 5 states the composite scores are standardized by the sd
# of those same 5 numbers, so a single outlier state inflates the sd and drags
# every other state's z toward zero, washing the map toward SIDEWAYS by a
# different amount in each fit. IT IS NOT RECOMMENDED UNTIL THE STANDARDIZATION
# IS FIXED (DIR_SCALE='mad' is the obvious first thing to try). DIRECTION_MODE
# stays 'rank'.
# ---------------------------------------------------------------------------
def _sigmoid(x):
    """Overflow-free logistic. exp is evaluated only on the non-positive side."""
    x = np.asarray(x, dtype=float)
    out = np.empty_like(x)
    p, n = x >= 0, x < 0
    out[p] = 1.0 / (1.0 + np.exp(-x[p]))
    e = np.exp(x[n])
    out[n] = e / (1.0 + e)
    return out


def soft_direction_weights(means, feature_subset, exclude=None, tau=None, c=None,
                           require_reachable=True, scale=None):
    """SOFT replacement for `direction_buckets`. Returns (W, z, scores).

    W is an (N, 3) row-stochastic matrix with columns [bull, side, bear]. The
    scorer is `composite_subset` -- the notebook's own, unchanged -- so FIX 1
    (`DIRECTION_EXCLUDE`) is threaded through untouched and 'soft' reads exactly
    the same evidence 'rank' does. Only the mapping score -> bucket changes: a
    step function of the RANK becomes a smooth function of the VALUE.

    Standardizing ACROSS STATES (not across bars) is what makes DIR_TAU / DIR_C
    scale-free -- and is also the weakness described in the block comment above.

    require_reachable : enforce that no bucket is structurally dead (the invariant
    the reverted sign+deadzone attempt violated). For c > 0 and tau > 0 this holds
    on every state in exact arithmetic; it can only fail when the sigmoids
    SATURATE in floating point, i.e. as tau -> 0, where the rule degenerates back
    into that deadzone.
    """
    scores = composite_subset(means, feature_subset, exclude)
    n_st = len(scores)
    assert n_st >= 3, 'need at least 3 states for a 3-bucket direction map'
    _scl = DIR_SCALE if scale is None else scale
    if _scl == 'sd':
        ctr, sd = float(np.mean(scores)), float(np.std(scores))
    else:
        assert _scl == 'mad', f'unknown DIR_SCALE {_scl!r}'
        ctr = float(np.median(scores))
        sd = 1.4826 * float(np.median(np.abs(scores - ctr)))
        if sd <= 0:                       # >= half the states tied: fall back
            ctr, sd = float(np.mean(scores)), float(np.std(scores))
    z = (scores - ctr) / (sd if sd > 0 else 1.0)

    tau = DIR_TAU if tau is None else float(tau)
    c = DIR_C if c is None else float(c)
    t = max(tau, 1e-12)                      # tau -> 0 becomes a hard threshold

    w_bull = _sigmoid((z - c) / t)
    w_bear = _sigmoid((-z - c) / t)
    w_side = np.maximum(0.0, 1.0 - w_bull - w_bear)

    W = np.stack([w_bull, w_side, w_bear], axis=1)
    tot = W.sum(axis=1)
    assert (tot > 0).all(), 'a state ended up with zero weight in all three buckets'
    W = W / tot[:, None]
    # EXACT partition: after the division the row sum is 1 only to within a ulp,
    # so the residual is handed to the row's LARGEST component (>= 1/3, so
    # `1 - rest` stays safely positive and non-negativity survives).
    for i in range(n_st):
        j = int(np.argmax(W[i]))
        others = [k for k in range(3) if k != j]
        W[i, j] = 1.0 - (W[i, others[0]] + W[i, others[1]])
    assert (np.abs(W.sum(axis=1) - 1.0) <= 4 * np.finfo(float).eps).all(), \
        'per-state weights must partition to 1'
    assert (W >= 0.0).all(), 'per-state weights must be non-negative'
    if require_reachable:
        assert W[:, 1].max() > 0.0, \
            'SIDE weight is zero on every state -> SIDEWAYS unreachable (the deadzone bug)'
        assert W[:, 0].max() > 0.0, 'BULL weight is zero on every state'
        assert W[:, 2].max() > 0.0, 'BEAR weight is zero on every state'
    return W, z, scores


def hard_direction_weights(means, feature_subset, exclude=None):
    """`direction_buckets` expressed as the same one-hot (N, 3) object
    `soft_direction_weights` returns, so the two modes are one construction with
    two settings rather than two code paths. Diagnostics only -- the 'rank'
    labeling path calls `ensemble_direction_masses` itself, so the bit-for-bit
    claim is never routed through this helper."""
    d = direction_buckets(means, feature_subset, exclude)
    W = np.zeros((len(d), 3), dtype=float)
    W[d == 1, 0] = 1.0
    W[d == 0, 1] = 1.0
    W[d == -1, 2] = 1.0
    assert (W.sum(axis=1) == 1.0).all()
    return W, d


def ensemble_direction_masses_by_mode(models, Xs, feature_subset, exclude=None,
                                      direction_mode=None, tau=None, c=None, scale=None):
    """`ensemble_direction_masses` with the state -> bucket map made switchable.

    mode='rank' : DELEGATES to the unchanged function above, so it is bit-for-bit
                  today's behaviour by construction rather than by
                  re-implementation.
    mode='soft' : identical pipeline -- CAUSAL filtered posteriors, per-model
                  collapse to 3 direction columns, average across the ensemble --
                  except the collapse is `probs @ W` instead of summing the
                  columns of a hard partition.

    Averaging across the ensemble stays valid for the same reason it always did:
    direction masses are PERMUTATION-INVARIANT. Causality is untouched: W depends
    on the FITTED MEANS only (fit window) and `probs` is the forward-only
    filtered posterior.
    """
    mode = DIRECTION_MODE if direction_mode is None else direction_mode
    if mode == 'rank':
        assert tau is None and c is None and scale is None, \
            'dir_tau / dir_c / dir_scale are soft-mode knobs and do nothing under rank'
        return ensemble_direction_masses(models, Xs, feature_subset, exclude)
    assert mode == 'soft', f'unknown DIRECTION_MODE {mode!r}'
    models = list(models)
    assert len(models) >= 1, 'ensemble must contain at least one model'
    acc = np.zeros((len(Xs), 3), dtype=float)      # columns: bull, side, bear
    probs0 = None
    for i, m in enumerate(models):
        W, _z, _sc = soft_direction_weights(m.means_, feature_subset, exclude, tau, c,
                                            scale=scale)
        probs = _filtered_posteriors(m, Xs)        # CAUSAL, forward-only
        if i == 0:
            probs0 = probs
        acc += probs @ W
    acc /= len(models)
    # INVARIANT: a convex combination of simplex rows is a simplex row.
    assert np.allclose(acc.sum(axis=1), 1.0, atol=1e-9), \
        'soft ensembled direction masses must partition to 1'
    return acc[:, 0], acc[:, 1], acc[:, 2], probs0


DIR_REACH_MIN = 0.10   # a direction bucket must carry at least this much mass on
                       # SOME bar to count as reachable. Deliberately a low bar:
                       # the point is to catch a STRUCTURALLY dead bucket (the
                       # deadzone bug), not to legislate an occupancy.


def label_bars(model, Xs, dates, price, trend_raw, n_fit, feature_subset,
               exclude=None, confirm_bars=None, z_exit=None, eff_exit=None,
               dir_feats=None, bar_dir_weight=None,
               intensity_mode=None, feat_raw=None, z_enter=None,
               target_rate=None, exit_slack=None, gate_hysteresis=None,
               direction_mode=None, dir_tau=None, dir_c=None, dir_scale=None,
               escalation_during_hold=None):
    """Direction + intensity + chop-filter labeling: an inline port of the
    engine's _fit_and_classify labeling block (no repo import).

    model         : fitted GaussianHMM, OR a list/tuple of them (a seed ensemble).
                    With a list, the bull/side/bear masses are the ENSEMBLE
                    AVERAGE; a single model (or a 1-element list) reproduces the
                    old single-fit behaviour bit for bit. Nothing else about the
                    labeling changes -- the gates, the CONF_L override, the
                    output columns and their semantics are identical either way.
    Xs            : scaled features for bars 0..len(dates)-1 (scaler fit on <= n_fit)
    dates         : DatetimeIndex for those bars
    price         : full close Series (reindexed internally; efficiency is causal)
    trend_raw     : TREND_FEATURE values aligned to `dates`
    n_fit         : number of LEADING bars the model/scaler were fit on -- the trend
                    mu/sd baseline is frozen on exactly this window (no look-ahead)

    The three switchable labeling fixes (all default to the module-level config;
    the values in brackets reproduce the PRE-FIX behaviour bit for bit):

    exclude       : FIX 1, features that do not vote on direction  [()]
    confirm_bars  : FIX 2, causal confirmation delay in bars       [1]
    z_exit,
    eff_exit      : FIX 3, gate de-escalation band                 [Z_HI, EFF_HI]
    dir_feats     : FIX 4, RAW feature frame aligned to `dates` (the per-bar
                    direction score reads BAR_DIR_FEATURES out of it)
    bar_dir_weight: FIX 4, blend weight on the per-bar masses      [0.0]

    And the three switches added in this consolidation (again, the bracketed
    value reproduces the PRE-CHANGE behaviour bit for bit -- asserted against a
    frozen verbatim copy of the old function in Section 7H-viii):

    intensity_mode: FIX 5, 'frozen_z' | 'vol_norm'                 ['frozen_z']
    feat_raw      : FIX 5, raw feature frame (vol_norm reads vol_2h out of it);
                    falls back to `dir_feats`, which is the same frame at every
                    call site that supplies one
    z_enter,
    z_exit        : FIX 5/3, EXPLICIT band overrides, expressed in the units of
                    whichever intensity_mode is in force. These are no longer
                    frozen_z-only knobs -- see `gate_band`
    target_rate,
    exit_slack    : FIX 5/3, band derived by FIT-WINDOW target occupancy; valid
                    under EITHER mode
    gate_hysteresis: FIX 3 as a mode-INDEPENDENT boolean. False collapses exit
                    onto enter in either mode, which is what FIX-3-off means
                                                                   [False]
    direction_mode: FIX 6, 'rank' | 'soft'                         ['rank']
    dir_tau,
    dir_c,
    dir_scale     : FIX 6 soft-mode knobs, rejected under 'rank'
    escalation_during_hold : 'allow' | 'block' | 'demote'          ['allow']

    Returns a DataFrame indexed by `dates` with the engine's column names.
    """
    models = list(model) if isinstance(model, (list, tuple)) else [model]

    # CAUSAL decoding: filtered (forward-only) posteriors, NOT hmmlearn's
    # forward-backward predict_proba / Viterbi predict -- both of those smooth
    # bar t with bars after t, which is look-ahead in a trading label.
    #
    # Aggregate probability mass BY DIRECTION BUCKET, not by individual state: a
    # bull move split across two bullish states must not be diluted below CONF_L
    # and mislabelled SIDEWAYS. With an ensemble, those bucket masses are then
    # AVERAGED over the K fits (permutation-invariant, so this is well defined).
    # FIX 6 rides here: 'rank' delegates to the unchanged ensemble function, so
    # the default path is bit-for-bit what it was.
    _dmode = DIRECTION_MODE if direction_mode is None else direction_mode
    bull_mass, side_mass, bear_mass, probs = ensemble_direction_masses_by_mode(
        models, Xs, feature_subset, exclude, _dmode, dir_tau, dir_c, dir_scale)
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6), \
        'direction masses must partition to 1'

    # ---- FIX 4: blend in the CAUSAL PER-BAR direction ----------------------
    # The masses above are per-STATE evidence: bar t inherits the direction of
    # whichever states it sits in, and a directionally MIXED state (the classic
    # high-volatility state, which holds both sharp selloffs and sharp rallies)
    # hands the same answer to bars pointing opposite ways. The per-bar score is
    # computed from signed features only and asks the question one bar at a time.
    #
    # A convex combination of two 3-simplex points is a 3-simplex point, so the
    # partition-to-1 invariant survives untouched, and so does everything built
    # on it (prob_*, confidence, the CONF_L override, the gates, the hysteresis).
    #
    # w = 0.0 is EXACT: 1.0*m + 0.0*b == m in IEEE754 for finite non-negative m.
    _bw = BAR_DIR_WEIGHT if bar_dir_weight is None else float(bar_dir_weight)
    assert 0.0 <= _bw <= 1.0, 'BAR_DIR_WEIGHT must be in [0, 1]'
    if dir_feats is None:
        assert _bw == 0.0, \
            'bar_dir_weight > 0 requires dir_feats (the raw feature frame for these bars)'
        bar_score = np.full(len(dates), np.nan)
    else:
        assert len(dir_feats) == len(dates), 'dir_feats must be aligned 1:1 with dates'
        bar_score = bar_direction_score(dir_feats, n_fit)
        _bb, _bs, _br = bar_direction_masses(bar_score)
        bull_mass = (1.0 - _bw) * bull_mass + _bw * _bb
        side_mass = (1.0 - _bw) * side_mass + _bw * _bs
        bear_mass = (1.0 - _bw) * bear_mass + _bw * _br
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6), \
        'BLENDED direction masses must still partition to 1'

    # `states` is the FIRST ensemble member's filtered argmax. Raw state indices
    # have no ensemble-wide meaning (they permute between fits), so this column
    # is reporting-only and is never used to form a label. With K=1 it is exactly
    # the old hmm_state_int.
    states = probs.argmax(axis=1)

    # REACHABILITY -- no direction bucket may be structurally dead. This is the
    # invariant the reverted sign+deadzone attempt violated (empty side bucket ->
    # SIDEWAYS unreachable). Checked under BOTH direction modes, on the BLENDED
    # masses, i.e. on what the labels are actually formed from.
    for _nm, _m in (('BULL', bull_mass), ('SIDE', side_mass), ('BEAR', bear_mass)):
        assert float(np.max(_m)) >= DIR_REACH_MIN, \
            f'{_nm} bucket never reaches {DIR_REACH_MIN} mass on any bar -> unreachable'

    n = len(dates)
    eff = trend_efficiency(price.reindex(dates), EFF_WIN).values

    # ---- DIRECTION IS DECIDED FIRST -------------------------------------
    # The intensity gate is computed AFTER the direction now, because
    # ESCALATION_DURING_HOLD needs to know whether the emitted direction is
    # contested before it can decide whether an escalation is allowed. Nothing
    # about the pre-change computation depended on the old order: dir_raw and
    # dir_emit never read the gate, and the gate never read the direction. With
    # ESCALATION_DURING_HOLD='allow' the reorder is a pure no-op, and Section
    # 7H-viii asserts that bit-for-bit against the frozen old function.
    regime_confidence = np.maximum(np.maximum(bull_mass, bear_mass), side_mass)
    masses = np.stack([bull_mass, bear_mass, side_mass], axis=1)
    winner = masses.argmax(axis=1)              # 0=bull, 1=bear, 2=side (argmax, not "nonzero")

    # DIRECTION first (bull / bear / side), including the CONF_L override, ...
    dir_raw = np.select([winner == 0, winner == 1, winner == 2],
                        ['BULL', 'BEAR', 'SIDE'], default='SIDE')
    dir_raw = np.where(regime_confidence < CONF_L, 'SIDE', dir_raw)

    # ... then FIX 2, the causal confirmation delay, applied to the DIRECTION.
    #
    # Why direction and not the full 5-label string: an H<->L intensity flicker
    # inside one direction would otherwise keep resetting the direction candidate
    # and can freeze the emitted label indefinitely (raw H_BULL, L_BULL, H_BULL,
    # L_BULL, ... never confirms anything at CONFIRM_BARS=2, so a clean rally
    # would stay stuck on whatever preceded it). Direction flicker is also
    # precisely the barcode the user objected to; H<->L flicker is fix 3's job.
    # confirm_bars=1 leaves dir_emit == dir_raw, i.e. the old behaviour exactly.
    dir_emit = confirm_delay(dir_raw, confirm_bars)

    # ---- THE INTENSITY GATE ---------------------------------------------
    # FIX 5: the trend-magnitude series AND the band it is graded against now
    # come from one place (`gate_band`), so INTENSITY_MODE and the enter/exit
    # band are independent knobs rather than mutually exclusive ones.
    #
    # FIX 3 lives inside that band: escalation requires |z| >= thr_enter AND
    # eff >= EFF_HI; de-escalation requires falling below the LOOSER exit band,
    # so a bar hovering at the threshold no longer flickers H/L every bar.
    # gate_hysteresis=False collapses exit onto enter in either mode, which is
    # the pre-FIX-3 memoryless test.
    _imode = INTENSITY_MODE if intensity_mode is None else intensity_mode
    _fr = feat_raw if feat_raw is not None else dir_feats
    z, thr_enter, thr_exit = gate_band(trend_raw, _fr, n_fit, _imode,
                                       z_enter, z_exit, target_rate, exit_slack,
                                       gate_hysteresis)
    assert thr_exit <= thr_enter, 'the exit band must not be tighter than the enter band'

    # ESCALATION_DURING_HOLD: a bar is CONTESTED when this bar's own evidence
    # (dir_raw) disagrees with the direction CONFIRM_BARS is holding (dir_emit).
    # Under 'allow' both masks are all-False and this is a no-op.
    _hold = ESCALATION_DURING_HOLD if escalation_during_hold is None else escalation_during_hold
    _blk, _fex, _contested = hold_masks(dir_raw, dir_emit, _hold)

    intens = intensity_state(z, eff, thr_enter, EFF_HI, thr_exit, eff_exit,
                             block_enter=_blk, force_exit=_fex)
    hi_bull = intens == 1
    hi_bear = intens == -1

    prob_cols = {
        'H_BULL':   np.where(hi_bull,  bull_mass, 0.0),
        'L_BULL':   np.where(~hi_bull, bull_mass, 0.0),
        'H_BEAR':   np.where(hi_bear,  bear_mass, 0.0),
        'L_BEAR':   np.where(~hi_bear, bear_mass, 0.0),
        'SIDEWAYS': side_mass,
    }
    # INVARIANT: the 5 prob buckets always partition the full probability mass,
    # independently of the confidence override above.
    assert np.allclose(sum(prob_cols.values()), 1.0, atol=1e-6), \
        'prob_* columns must sum to 1'

    # Intensity is then graded at bar t from the (hysteretic) gate state, so an
    # emitted H bar always clears the gates AT THAT BAR -- the chop-filter
    # invariant below is a statement about the label that is actually emitted.
    # np.full/boolean assignment rather than np.select: np.select would type the
    # result from the choicelist (<U6) and silently TRUNCATE 'SIDEWAYS' to
    # 'SIDEWA'. dtype is pinned explicitly here.
    def _compose(direction):
        st = np.full(n, 'SIDEWAYS', dtype='<U8')
        mb = direction == 'BULL'
        st[mb] = np.where(hi_bull, 'H_BULL', 'L_BULL')[mb]
        mr = direction == 'BEAR'
        st[mr] = np.where(hi_bear, 'H_BEAR', 'L_BEAR')[mr]
        return st

    regime_state = _compose(dir_emit)     # what the notebook uses everywhere
    raw_state    = _compose(dir_raw)      # pre-confirmation, diagnostics only

    # INVARIANT (chop filter), generalised for the enter/exit bands: no bar may be
    # graded H without clearing the gate that is ACTIVE for it -- the ENTER gate on
    # the first bar of an H run, the (looser) EXIT gate on a bar the run is being
    # held through. Stated against the thresholds ACTUALLY IN FORCE (thr_enter /
    # thr_exit), which is what makes it mode-independent; with hysteresis off the
    # two collapse into the single original assert.
    _ex = EFF_HI_EXIT if eff_exit is None else eff_exit
    is_h = np.isin(regime_state, ['H_BULL', 'H_BEAR'])
    if is_h.any():
        assert (eff[is_h] >= min(EFF_HI, _ex)).all(), 'H bar below the EFF exit band -> chop filter bypassed'
        assert (np.abs(z[is_h]) >= thr_exit).all(), 'H bar below the intensity exit band -> magnitude gate bypassed'
        # and every ESCALATION -- the bar on which the gate state machine turned ON,
        # which is where the ENTER band must have been cleared. (The bar an emitted
        # H *label* run starts on is NOT the right anchor: the direction can flip to
        # BULL several bars into an already-escalated stretch, and that bar only
        # owes the exit band.)
        _prev_i = np.concatenate(([0], intens[:-1]))
        _on = np.flatnonzero((intens != 0) & (intens != _prev_i))   # incl. +1 -> -1 flips
        assert (eff[_on] >= EFF_HI).all(), 'H escalation below EFF_HI -> enter gate bypassed'
        assert (np.abs(z[_on]) >= thr_enter).all(), \
            'H escalation below the intensity enter threshold -> enter gate bypassed'
        # ESCALATION_DURING_HOLD: under 'block'/'demote' no escalation may BEGIN on
        # a contested bar, and under 'demote' no H may be emitted on one at all.
        if _hold in ('block', 'demote'):
            assert not _contested[_on].any(), \
                'a NEW escalation fired on a contested bar -> ESCALATION_DURING_HOLD bypassed'
        if _hold == 'demote':
            assert not (is_h & _contested).any(), \
                'an H label survived on a contested bar under ESCALATION_DURING_HOLD=demote'
    assert set(np.unique(regime_state)).issubset(set(REGIME_LABELS)), 'unknown label emitted'

    out = pd.DataFrame({
        'tactical_regime_state':      regime_state,
        'tactical_regime_confidence': regime_confidence,
        'hmm_state_int':              states.astype(int),
        'prob_H_BULL':                prob_cols['H_BULL'],
        'prob_L_BULL':                prob_cols['L_BULL'],
        'prob_SIDEWAYS':              prob_cols['SIDEWAYS'],
        'prob_L_BEAR':                prob_cols['L_BEAR'],
        'prob_H_BEAR':                prob_cols['H_BEAR'],
        'trend_z':                    z,
        'trend_efficiency':           eff,
        # diagnostics for Section 7H (never inputs to anything):
        'gate_intensity':             intens,
        'regime_state_raw':           raw_state,
        'bar_dir_score':              bar_score,
        # ESCALATION_DURING_HOLD diagnostics (never inputs to anything).
        # `contested` keeps the prototype's column name so the two artifacts
        # can be diffed directly.
        'dir_raw':                    dir_raw,
        'dir_emit':                   dir_emit,
        'contested':                  _contested,
    }, index=dates)
    out.index.name = 'date'
    out.attrs['intensity_mode'] = _imode
    out.attrs['direction_mode'] = _dmode
    out.attrs['escalation_during_hold'] = _hold
    out.attrs['thr_enter'] = float(thr_enter)
    out.attrs['thr_exit'] = float(thr_exit)
    return out


# ---------------------------------------------------------------------------
# THE FROZEN PRE-CHANGE REFERENCE.
#
# `label_bars_legacy` is a VERBATIM copy of `label_bars` as it stood BEFORE this
# consolidation -- before INTENSITY_MODE, DIRECTION_MODE, ESCALATION_DURING_HOLD
# and the direction-before-intensity reorder. It exists for exactly one purpose:
# Section 7H-viii runs both functions over a matrix of argument combinations and
# asserts BIT-FOR-BIT equality of every emitted column whenever the new switches
# sit at their OFF values. That turns "these switches are no-ops when off" from a
# claim in a comment into a test.
#
# It is never called by the pipeline. Do not "improve" it -- its whole value is
# that it is frozen.
# ---------------------------------------------------------------------------
def label_bars_legacy(model, Xs, dates, price, trend_raw, n_fit, feature_subset,
                      exclude=None, confirm_bars=None, z_exit=None, eff_exit=None,
                      dir_feats=None, bar_dir_weight=None):
    models = list(model) if isinstance(model, (list, tuple)) else [model]

    bull_mass, side_mass, bear_mass, probs = \
        ensemble_direction_masses(models, Xs, feature_subset, exclude)
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6)

    _bw = BAR_DIR_WEIGHT if bar_dir_weight is None else float(bar_dir_weight)
    assert 0.0 <= _bw <= 1.0
    if dir_feats is None:
        assert _bw == 0.0
        bar_score = np.full(len(dates), np.nan)
    else:
        assert len(dir_feats) == len(dates)
        bar_score = bar_direction_score(dir_feats, n_fit)
        _bb, _bs, _br = bar_direction_masses(bar_score)
        bull_mass = (1.0 - _bw) * bull_mass + _bw * _bb
        side_mass = (1.0 - _bw) * side_mass + _bw * _bs
        bear_mass = (1.0 - _bw) * bear_mass + _bw * _br
    assert np.allclose(bull_mass + bear_mass + side_mass, 1.0, atol=1e-6)

    states = probs.argmax(axis=1)

    trend_raw = np.asarray(trend_raw, dtype=float)
    trend_mu = float(np.mean(trend_raw[:n_fit]))
    trend_sd = float(np.std(trend_raw[:n_fit]))
    z = (trend_raw - trend_mu) / (trend_sd if trend_sd > 0 else 1.0)

    eff = trend_efficiency(price.reindex(dates), EFF_WIN).values
    intens = intensity_state(z, eff, Z_HI, EFF_HI, z_exit, eff_exit)
    hi_bull = intens == 1
    hi_bear = intens == -1

    n = len(dates)
    prob_cols = {
        'H_BULL':   np.where(hi_bull,  bull_mass, 0.0),
        'L_BULL':   np.where(~hi_bull, bull_mass, 0.0),
        'H_BEAR':   np.where(hi_bear,  bear_mass, 0.0),
        'L_BEAR':   np.where(~hi_bear, bear_mass, 0.0),
        'SIDEWAYS': side_mass,
    }
    assert np.allclose(sum(prob_cols.values()), 1.0, atol=1e-6)

    regime_confidence = np.maximum(np.maximum(bull_mass, bear_mass), side_mass)
    masses = np.stack([bull_mass, bear_mass, side_mass], axis=1)
    winner = masses.argmax(axis=1)

    dir_raw = np.select([winner == 0, winner == 1, winner == 2],
                        ['BULL', 'BEAR', 'SIDE'], default='SIDE')
    dir_raw = np.where(regime_confidence < CONF_L, 'SIDE', dir_raw)
    dir_emit = confirm_delay(dir_raw, confirm_bars)

    def _compose(direction):
        st = np.full(n, 'SIDEWAYS', dtype='<U8')
        mb = direction == 'BULL'
        st[mb] = np.where(hi_bull, 'H_BULL', 'L_BULL')[mb]
        mr = direction == 'BEAR'
        st[mr] = np.where(hi_bear, 'H_BEAR', 'L_BEAR')[mr]
        return st

    regime_state = _compose(dir_emit)
    raw_state    = _compose(dir_raw)

    return pd.DataFrame({
        'tactical_regime_state':      regime_state,
        'tactical_regime_confidence': regime_confidence,
        'hmm_state_int':              states.astype(int),
        'prob_H_BULL':                prob_cols['H_BULL'],
        'prob_L_BULL':                prob_cols['L_BULL'],
        'prob_SIDEWAYS':              prob_cols['SIDEWAYS'],
        'prob_L_BEAR':                prob_cols['L_BEAR'],
        'prob_H_BEAR':                prob_cols['H_BEAR'],
        'trend_z':                    z,
        'trend_efficiency':           eff,
        'gate_intensity':             intens,
        'regime_state_raw':           raw_state,
        'bar_dir_score':              bar_score,
    }, index=dates)


print('features + direction/intensity labeling core ready')
print(f'  gate      : INTENSITY_MODE={INTENSITY_MODE!r}  band via gate_band() '
      f'(mode and band are independent knobs)')
print(f'  direction : DIRECTION_MODE={DIRECTION_MODE!r}   hold policy: '
      f'ESCALATION_DURING_HOLD={ESCALATION_DURING_HOLD!r}')
print('  label_bars_legacy (frozen pre-change copy) available for the 7H-viii equivalence test')

In [ ]:
# ==========================================================================
# ENGINE CELL 4/4 -- regime-background plot helpers (VERBATIM)
# ==========================================================================
def regime_blocks(series):
    """[(label, start_ts, end_ts), ...] contiguous runs of the same label."""
    vals, idx = series.values, series.index
    blocks, start = [], 0
    for i in range(1, len(vals)):
        if vals[i] != vals[i - 1]:
            blocks.append((vals[start], idx[start], idx[i - 1]))
            start = i
    blocks.append((vals[start], idx[start], idx[-1]))
    return blocks


def shade_bands(ax, spans, alpha=0.35, zorder=1):
    """Paint (label, x0, x1) spans as ONE PolyCollection PER LABEL.

    Replaces a per-block `ax.axvspan` loop. On this data a chart has 500+
    contiguous regime blocks, so the loop built 500+ individual Patch artists per
    axes and ~13 figures paid for it; this builds at most len(REGIME_LABELS)
    collections instead, with the same geometry.

    Visually identical, by construction rather than by eye:
      * the x-ranges come from the SAME `regime_blocks` output, unchanged;
      * `broken_barh` with `ax.get_xaxis_transform()` is the same blended
        transform `axvspan` uses -- x in DATA coordinates, y in AXES fraction
        0..1 -- so bands span the full height and ignore the y data limits
        exactly as axvspan did;
      * colour, alpha, linewidth=0 and zorder=1 are the axvspan values.

    Grouping by label is safe because `regime_blocks` returns DISJOINT spans, so
    no two bands overlap and the draw order between them cannot matter.

    ------------------------------------------------------------------------
    THE Y-AXIS BUG THIS FIXES (user-visible; NOT reproducible on every
    matplotlib, so it is fixed structurally rather than by chasing a repro).
    ------------------------------------------------------------------------
    On the user's Kaggle matplotlib the shaded price panels came out with the
    y-axis dragged down to 0, squashing the price line into the top fifth of the
    panel. The cause is the blended transform: the band geometry is y = 0..1 in
    AXES-FRACTION coordinates, but `broken_barh` -> `add_collection` defaults to
    `autolim=True`, and an older matplotlib folds the collection's raw y-extent
    (those literal 0 and 1) into the axes DATA limits before the transform is
    considered. The autoscaler then has to fit both `[0, 1]` and `[24000, 26000]`
    and produces `[0, 26000]`.

    Fixed two ways, deliberately belt-and-braces:
      1. HERE -- build the PolyCollection directly and add it with
         `autolim=False`, so it cannot contribute to the datalim on ANY
         matplotlib version. This is the structural fix.
      2. At every call site -- `set_price_ylim` sets the y-limits EXPLICITLY from
         the plotted series, so the autoscaler is never consulted at all.
    Each of the two alone is sufficient; together the panel cannot regress.

    The speed optimization is NOT reverted: this still builds at most
    len(REGIME_LABELS) collections per axes, not one Patch per block.
    """
    import matplotlib.dates as _mdates
    from matplotlib.collections import PolyCollection
    by_lab = {}
    for lb, d0, d1 in spans:
        x0, x1 = _mdates.date2num(d0), _mdates.date2num(d1)
        by_lab.setdefault(lb, []).append((x0, x1))
    for lb, xr in by_lab.items():
        verts = [[(x0, 0.0), (x1, 0.0), (x1, 1.0), (x0, 1.0)] for x0, x1 in xr]
        coll = PolyCollection(verts,
                              facecolors=REGIME_COLORS.get(lb, '#808080'),
                              alpha=alpha, linewidths=0, zorder=zorder)
        # x in DATA coords, y in AXES fraction 0..1 -- the same blended transform
        # axvspan and broken_barh use, so the bands still span the full height.
        coll.set_transform(ax.get_xaxis_transform())
        ax.add_collection(coll, autolim=False)     # <-- cannot touch the datalim


def shade_regimes(ax, series, alpha=0.35):
    shade_bands(ax, regime_blocks(series), alpha=alpha)


# Registry of every shaded panel whose y-limits were set explicitly, so Section
# 7Z can assert -- once, centrally -- that each one brackets its own series and
# excludes 0. A panel that forgot to call this simply never gets checked, so the
# registry is printed with its expected count too.
YLIM_CHECKS = []


def set_price_ylim(ax, series, pad=0.03, tag=''):
    """Set y-limits EXPLICITLY from the plotted series and record the check.

    Never leaves a shaded price/VIX panel to the autoscaler. `pad` is a fraction
    of the series range (falling back to a fraction of the level, then to 1.0,
    for a degenerate flat series).
    """
    v = np.asarray(series, dtype=float)
    v = v[np.isfinite(v)]
    assert v.size, f'set_price_ylim got no finite values ({tag})'
    lo, hi = float(v.min()), float(v.max())
    m = (hi - lo) * pad or abs(hi) * pad or 1.0
    ax.set_ylim(lo - m, hi + m)
    YLIM_CHECKS.append((tag, ax, lo, hi))
    return lo, hi


SPLIT_STYLE = dict(color='blue', linestyle='--', linewidth=1.2, zorder=5)
SPLIT_LABEL = 'train/test split'


def mark_split(ax, index, split_ts):
    """Draw the anchored train/test boundary, if it falls inside this panel.

    Returns the legend handle when the line was drawn and None when the panel's
    window does not contain the split (the zoom panels), so a caller can add the
    legend entry only where there is actually a line to explain.

    THE SPLIT IS A BACKTEST DEVICE. It marks where the fit window ended so that
    bars to its right can be scored on data the model never saw. A LIVE engine
    has no such boundary: it fits on all history to date and classifies the next
    bar. Nothing to the left of this line is "less real" -- it is simply
    in-sample, and therefore not evidence.
    """
    if split_ts is None or len(index) == 0:
        return None
    if not (index[0] <= split_ts <= index[-1]):
        return None
    ax.axvline(split_ts, **SPLIT_STYLE)
    return plt.Line2D([0], [0], color=SPLIT_STYLE['color'],
                      ls=SPLIT_STYLE['linestyle'], label=SPLIT_LABEL)


def regime_legend(ax, loc='upper left', extra=None, **kw):
    handles = [mpatches.Patch(color=REGIME_COLORS[r], alpha=0.7, label=r) for r in REGIME_LABELS]
    if extra:
        handles += extra
    ax.legend(handles=handles, loc=loc, fontsize=8, ncol=3, **kw)


def synth_tag():
    return "  [SYNTHETIC DATA - illustrative only]" if TAC_SYNTH else "  [real yfinance data]"

In [ ]:
import contextlib, time, os, io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FIGDIR = 'sw_figs'
os.makedirs(FIGDIR, exist_ok=True)
FIGS = []

_emdbm_raw = ensemble_direction_masses_by_mode      # the engine's own, kept for reference

print('engine loaded. shipped labeling constants:')
for _k in ('CONF_L', 'CONFIRM_BARS', 'ENSEMBLE_K', 'BASE_SEED', 'DIRECTION_MODE',
           'DIRECTION_EXCLUDE', 'ESCALATION_DURING_HOLD', 'BAR_DIR_WEIGHT',
           'TRAIN_FRACTION', 'EFF_WIN', 'EFF_HI', 'INTENSITY_MODE',
           'H_TARGET_RATE', 'ADOPTED_CONFIG_NAME'):
    print(f'  {_k:<24} = {globals()[_k]!r}')
CFG_BY_NAME = {c['name']: c for c in CONFIGS}
BASE_CFG = CFG_BY_NAME[ADOPTED_CONFIG_NAME]
BASE_FEATURES = tuple(BASE_CFG['features'])
BASE_N, BASE_COV = BASE_CFG['N'], BASE_CFG['cov']
print(f'  adopted config           = {BASE_CFG}')
assert BAR_DIR_WEIGHT == 0.0, 'BAR_DIR_WEIGHT is fixed at 0.0 for this study'

## 2. Data

`yfinance` and `stooq` are blocked by the proxy in this environment (HTTP 403),
so the user's own 2h `^NSEI` series cannot be reproduced here. Two series are
used instead:

**A — REAL daily `NIFTY 50` OHLC, fetched from `raw.githubusercontent.com`.**
The user explicitly authorised fetching market data from GitHub.

> ### ⚠ GITHUB DATA — UNVERIFIED PROVENANCE, machinery-grade not decision-grade
> This file was not obtained from NSE or from a licensed vendor. It is sanity-
> checked below (monotone unique dates, positive prices, OHLC consistency, and
> every >5% single-day move matched to a known real event) and it passes — but
> its provenance is **unverified**. Every number derived from it carries this
> label. **The user's Kaggle run on the real 2h series remains the source of
> truth for the final verdict.**

**Cadence caveat, stated plainly.** These are **daily** bars, not 2h. The
engine's feature lookbacks are expressed in *bars*, so the features are
well-formed, but the absolute occupancy will not match the 2h run: the baseline
here comes out at **≈29% SIDEWAYS**, not 34.7%. That is fine for this study,
because every candidate is compared against **the baseline measured on the same
data** — the question asked is *relative*. Bar count (2744) is close to the 2h
run's 2894, so `W` (per-100-bars) and rows-per-parameter are comparable.

**B — the master's own synthetic 2h generator**, cadence-matched, as a
robustness cross-check. It earns its place in section 7: it is what exposed the
fact that a *fixed* margin is not scale-free.

In [ ]:
NIFTY_CSV_URL = ('https://raw.githubusercontent.com/NupurBachhuka/financial_forecast_v2/'
                 'main/rawdata/NIFTY50_2015_2026_merged.csv')

def load_github_daily():
    print(f'FETCH: {NIFTY_CSV_URL}')
    with urllib.request.urlopen(NIFTY_CSV_URL, timeout=90) as fh:
        raw = fh.read().decode('utf-8')
    d = pd.read_csv(io.StringIO(raw))
    d['Date'] = pd.to_datetime(d['Date'])
    d = d.sort_values('Date').reset_index(drop=True)

    # ---- SANITY CHECKS. A series that fails any of these is NOT used. --------
    print(f'  rows            : {len(d)}')
    print(f'  span            : {d.Date.min():%Y-%m-%d} -> {d.Date.max():%Y-%m-%d}')
    assert d['Date'].is_monotonic_increasing, 'dates not monotone'
    print('  monotone dates  : OK')
    assert not d['Date'].duplicated().any(), 'duplicate dates'
    print('  duplicate dates : none')
    ohlc = d[['Open', 'High', 'Low', 'Close']]
    assert (ohlc > 0).all().all(), 'non-positive price'
    print('  non-positive px : none')
    assert not ohlc.isna().any().any(), 'NaN price'
    bad = ((d.High < ohlc[['Open', 'Close', 'Low']].max(axis=1)) |
           (d.Low > ohlc[['Open', 'Close', 'High']].min(axis=1))).sum()
    assert bad == 0, f'{bad} OHLC-inconsistent rows'
    print('  OHLC consistency: OK')

    r = d.Close.pct_change()
    big = d.loc[r.abs() > 0.05, ['Date', 'Close']].assign(ret=r[r.abs() > 0.05])
    print(f'  |1-day move| > 5%: {len(big)} rows, max {r.abs().max():.2%} on '
          f'{d.Date[r.abs().idxmax()]:%Y-%m-%d}')
    for _, row in big.iterrows():
        print(f'      {row.Date:%Y-%m-%d}  {row.ret:+.2%}')
    print('  -> every one of the above is a KNOWN REAL EVENT: 2015-08-24 China')
    print('     "Black Monday"; 2019-09-20 corporate-tax-cut rally; the 2020-03/04')
    print('     COVID crash and rebound; 2024-06-04 Indian election result day.')
    print('     No unexplained spike remains -> series accepted.')

    px = pd.Series(d['Close'].values, index=pd.DatetimeIndex(d['Date']), name='nifty')
    # This file carries no India VIX column, and the engine needs one for vix_chg.
    # A CAUSAL 20-bar trailing realised-vol proxy is substituted -- declared here,
    # not hidden. It is backward-looking only, so it introduces no look-ahead.
    lr = np.log(px / px.shift(1))
    vx = (lr.rolling(20).std() * np.sqrt(252) * 100).bfill().clip(6, 90)
    vx.name = 'vix'
    print('  VIX             : no VIX column in this file -> CAUSAL 20-bar trailing')
    print('                    realised-vol proxy substituted (declared, not hidden)')
    return px, vx


try:
    NIFTY_D, VIX_D = load_github_daily()
    REAL_OK = True
except Exception as e:
    print(f'\n!!! GitHub fetch FAILED ({e}). Falling back to SYNTHETIC ONLY.')
    REAL_OK = False

print()
print('*' * 78)
if REAL_OK:
    print('* GITHUB DATA - UNVERIFIED PROVENANCE, machinery-grade not decision-grade  *')
    print('* The Kaggle run on the real 2h series remains the source of truth.        *')
else:
    print('* NO TRUSTWORTHY REAL SERIES OBTAINED -> SYNTHETIC ONLY. Everything below  *')
    print('* is machinery-proof only and carries no verdict.                          *')
print('*' * 78)

NIFTY_S, VIX_S, _synth_flag = _synth()
assert _synth_flag
print(f'\nsynthetic 2h cross-check series: {len(NIFTY_S)} bars '
      f'{NIFTY_S.index[0]:%Y-%m-%d} -> {NIFTY_S.index[-1]:%Y-%m-%d}')

In [ ]:
# ===========================================================================
# DATASET container -- features, scaling, anchored fit window, cached fits.
# ===========================================================================
from sklearn.preprocessing import StandardScaler

class DS:
    def __init__(self, name, nifty, vix, tag):
        self.name, self.tag = name, tag
        self.nifty, self.vix = nifty, vix
        self.feat = build_features(nifty, vix, LOOKBACK_SCALE)
        self.dates = self.feat.index
        self.close = nifty.reindex(self.dates).values
        self.trend_raw = self.feat[TREND_FEATURE].values
        self.n = len(self.dates)
        self.n_fit = max(int(self.n * TRAIN_FRACTION), 50)
        self._xs, self._fits = {}, {}

    def X(self, features):
        k = tuple(features)
        if k not in self._xs:
            raw = self.feat[list(features)].values
            sc = StandardScaler().fit(raw[:self.n_fit])
            self._xs[k] = sc.transform(raw)
        return self._xs[k]

    def fits(self, base_seed, features=None, N=None, cov=None, K=None):
        features = BASE_FEATURES if features is None else features
        N = BASE_N if N is None else N
        cov = BASE_COV if cov is None else cov
        K = ENSEMBLE_K if K is None else K
        key = (base_seed, tuple(features), N, cov, K)
        if key not in self._fits:
            ms, _ = fit_hmm_ensemble(self.X(features)[:self.n_fit], N, cov,
                                     K=K, base_seed=base_seed)
            self._fits[key] = ms
        return self._fits[key]


DSETS = []
if REAL_OK:
    DSETS.append(DS('real_daily', NIFTY_D, VIX_D,
                    'GITHUB DATA - UNVERIFIED PROVENANCE, machinery-grade not decision-grade'))
DSETS.append(DS('synth_2h', NIFTY_S, VIX_S, 'SYNTHETIC - machinery-proof only'))
DS_BY_NAME = {d.name: d for d in DSETS}
PRIMARY = DSETS[0]

for d in DSETS:
    print(f'{d.name:<12} {d.n:>5} feature bars  {d.dates[0]:%Y-%m-%d} -> '
          f'{d.dates[-1]:%Y-%m-%d}  anchored fit window = {d.n_fit} '
          f'({TRAIN_FRACTION:.0%})')
    print(f'             {d.tag}')
print(f'\nPRIMARY dataset for the verdict: {PRIMARY.name}')

## 3. The rule hook — and its OFF value asserted bit-for-bit

`label_bars` decides direction like this (engine source, unedited):

```python
regime_confidence = np.maximum(np.maximum(bull_mass, bear_mass), side_mass)
masses  = np.stack([bull_mass, bear_mass, side_mass], axis=1)
winner  = masses.argmax(axis=1)
dir_raw = np.select([winner == 0, winner == 1, winner == 2], ['BULL','BEAR','SIDE'])
dir_raw = np.where(regime_confidence < CONF_L, 'SIDE', dir_raw)
```

Read it carefully: **it is already MAP** — argmax of the three bucket masses —
with `CONF_L` bolted on as a floor. So `R1` (MAP) is not a new rule at all; it is
the shipped rule with the floor removed.

Every candidate here has the same shape: *a function of the three bucket masses
returning `BULL`/`SIDE`/`BEAR`*. Rather than fork `label_bars`, a candidate is
installed by overriding `ensemble_direction_masses_by_mode` — the same device
`build_stability_lag.py`'s `S3` arm uses — and pinning the engine's own `CONF_L`
to `0.0` so it adds no second floor on top of the rule:

* on bars where the rule **agrees** with `argmax(masses)`, the **original masses
  pass through untouched**, so `prob_*` and `tactical_regime_confidence` stay
  genuine;
* on bars where the rule **overrides** `argmax`, the exact simplex vertex for the
  decided direction is emitted — an honest statement that on that bar the label
  is the rule's verdict rather than the raw posterior's.

Everything downstream is the **untouched engine**: `confirm_delay`, the intensity
gate and its hysteresis, the `prob_*` partition and its sum-to-1 assert, the
chop-filter invariant, the reachability asserts.

**The OFF value.** `R0` = this harness carrying `dec_conf(0.50)` must reproduce
the untouched engine *bit-for-bit*. That is asserted below on every dataset
before any candidate is measured. If it did not hold, nothing in this notebook
would mean anything.

In [ ]:
# ===========================================================================
# THE RULE HOOK.
# ===========================================================================
@contextlib.contextmanager
def _patched(mass_fn, conf_l):
    g = globals()
    old_fn, old_cl = g['ensemble_direction_masses_by_mode'], g['CONF_L']
    g['ensemble_direction_masses_by_mode'], g['CONF_L'] = mass_fn, conf_l
    try:
        yield
    finally:
        g['ensemble_direction_masses_by_mode'], g['CONF_L'] = old_fn, old_cl


def make_decide_masses(decide_fn):
    '''decide_fn(bull, side, bear) -> array of BULL/SIDE/BEAR.'''
    def fn(models, Xs, feature_subset, exclude, mode, tau, c, scale):
        b, s, r, probs = _emdbm_raw(models, Xs, feature_subset, exclude,
                                    mode, tau, c, scale)
        d = np.asarray(decide_fn(b, s, r))
        arg = np.array(['BULL', 'BEAR', 'SIDE'])[np.stack([b, r, s], 1).argmax(1)]
        same = d == arg
        return (np.where(same, b, (d == 'BULL').astype(float)),
                np.where(same, s, (d == 'SIDE').astype(float)),
                np.where(same, r, (d == 'BEAR').astype(float)), probs)
    return fn


def make_state_masses(state_fn):
    '''R4 only -- the decision is made on the STATE argmax, not on bucket mass,
    so no original mass can be passed through and the emitted masses are the
    rule's decision indicators. R4 is a reference arm, not a shipping candidate;
    its prob_* / confidence columns are therefore NOT posteriors and are not
    interpreted anywhere below.'''
    def fn(models, Xs, feature_subset, exclude, mode, tau, c, scale):
        _, _, _, probs = _emdbm_raw(models, Xs, feature_subset, exclude,
                                    mode, tau, c, scale)
        d = np.asarray(state_fn(models, Xs, feature_subset, exclude))
        return ((d == 'BULL').astype(float), (d == 'SIDE').astype(float),
                (d == 'BEAR').astype(float), probs)
    return fn


# ---------------------------------------------------------------------------
# THE RULE LIBRARY. Each returns a decide_fn.
# ---------------------------------------------------------------------------
def _argmax_dir(b, s, r):
    return np.array(['BULL', 'BEAR', 'SIDE'])[np.stack([b, r, s], 1).argmax(1)]


def dec_conf(floor):
    '''R0 / R1 / R2 / R3 -- argmax with an ABSOLUTE floor on the winning mass.
    floor=0.50 IS the shipped rule. floor=0.0 is pure MAP.'''
    def f(b, s, r):
        return np.where(np.stack([b, r, s], 1).max(1) < floor, 'SIDE',
                        _argmax_dir(b, s, r))
    return f


def dec_topmargin(m):
    '''R5 -- emit the argmax only if it beats the RUNNER-UP by more than m.'''
    def f(b, s, r):
        M = np.sort(np.stack([b, r, s], 1), axis=1)
        return np.where((M[:, -1] - M[:, -2]) <= m, 'SIDE', _argmax_dir(b, s, r))
    return f


def dec_sidemargin(m):
    '''R6 -- SIDE must EARN the label. Emit SIDE only when the side bucket beats
    the better of the two directional buckets by more than m; otherwise emit
    that directional bucket. m = 0 is exactly MAP. This is the first rule in the
    list that can actually LOWER SIDEWAYS, because it is the first one that
    asks SIDE to prove anything.'''
    def f(b, s, r):
        return np.where(s - np.maximum(b, r) > m, 'SIDE',
                        np.where(b >= r, 'BULL', 'BEAR'))
    return f


def dec_dirmargin(m):
    '''R7 -- SIDEWAYS as "bull and bear evidence are BALANCED". Direction is a
    TWO-way contest between the bull and bear buckets; the side bucket does not
    compete, it only sets the width of the dead zone.'''
    def f(b, s, r):
        return np.where(np.abs(b - r) <= m, 'SIDE',
                        np.where(b > r, 'BULL', 'BEAR'))
    return f


def dec_side_target(rate, n_fit):
    '''R10 -- R6 with the margin DERIVED, not guessed.

    A fixed absolute margin is not scale-free (section 7 shows m=0.40 giving 20%
    SIDEWAYS on the real series and 1.5% on the synthetic one). The engine
    already owns the right device for this: H_TARGET_RATE derives the intensity
    band from the FIT WINDOW at a target occupancy. Do the same here -- take the
    margin as the (1 - rate) quantile of  delta = side - max(bull, bear)  over
    the FIT WINDOW ONLY.

    CAUSALITY: the threshold is a constant computed from bars [0, n_fit), i.e.
    exactly the bars the HMM was trained on. No bar > t enters bar t's label.
    Identical in kind to the frozen trend_z baseline and to H_TARGET_RATE.'''
    def f(b, s, r):
        delta = s - np.maximum(b, r)
        thr = float(np.quantile(delta[:n_fit], 1.0 - rate))
        f.last_thr = thr
        return np.where(delta > thr, 'SIDE', np.where(b >= r, 'BULL', 'BEAR'))
    return f


def dec_dir_target(rate, n_fit):
    '''R11 -- the two-way analogue of R10: SIDEWAYS is the fit-window `rate`
    narrowest |bull - bear|. Same causality argument.'''
    def f(b, s, r):
        g = np.abs(b - r)
        thr = float(np.quantile(g[:n_fit], rate))
        f.last_thr = thr
        return np.where(g <= thr, 'SIDE', np.where(b > r, 'BULL', 'BEAR'))
    return f


def state_rule_R4(floor):
    '''R4 -- the ORIGINAL pre-V1.1 engine's rule, ported from
    regime_engine_tactical.py: argmax STATE, mapped to a direction by
    composite-bullishness RANK, with a max-STATE-probability confidence
    override. Two ports were forced and are declared:
      * the filtered (causal) posterior is used, not hmmlearn's smoothed
        predict_proba -- the original used the smoothed one, which is look-ahead
        and this project has already removed it everywhere else;
      * raw state indices permute between ensemble members, so the per-member
        decisions are combined by MAJORITY VOTE rather than averaged.'''
    def f(models, Xs, feature_subset, exclude):
        votes = np.zeros((len(Xs), 3))                  # bull, side, bear
        for m in list(models):
            p = _filtered_posteriors(m, Xs)
            dirs = direction_buckets(m.means_, feature_subset, exclude)
            d = dirs[p.argmax(axis=1)]
            d = np.where(p.max(axis=1) < floor, 0, d)
            votes[np.arange(len(d)), np.where(d == 1, 0, np.where(d == 0, 1, 2))] += 1
        return np.array(['BULL', 'SIDE', 'BEAR'])[votes.argmax(axis=1)]
    return f


def dec_all_side(b, s, r):
    '''The DEGENERATE STUB used to drive guard G4 in section 5.'''
    return np.full(len(b), 'SIDE')


print('rule hook ready. families: R0/R1/R2/R3 dec_conf | R4 state_rule_R4 | '
      'R5 dec_topmargin\n                          R6 dec_sidemargin | '
      'R7 dec_dirmargin | R10 dec_side_target | R11 dec_dir_target')

In [ ]:
# ===========================================================================
# LABELLING ENTRY POINTS. Both call the module-level `label_bars`, so the
# section-5 leakage tripwire (which SHADOWS that name) applies to every call.
# ===========================================================================
def label_decide(ds, models, decide_fn, features=None, **labkw):
    features = BASE_FEATURES if features is None else features
    kw = dict(labkw); kw.setdefault('dir_feats', ds.feat)
    with _patched(make_decide_masses(decide_fn), 0.0):
        return label_bars(models, ds.X(features), ds.dates, ds.nifty,
                          ds.trend_raw, ds.n_fit, list(features), **kw)


def label_state(ds, models, state_fn, features=None, **labkw):
    features = BASE_FEATURES if features is None else features
    kw = dict(labkw); kw.setdefault('dir_feats', ds.feat)
    with _patched(make_state_masses(state_fn), 0.0):
        return label_bars(models, ds.X(features), ds.dates, ds.nifty,
                          ds.trend_raw, ds.n_fit, list(features), **kw)


def label_untouched(ds, models, features=None, **labkw):
    '''The engine exactly as shipped -- no override, CONF_L as configured.'''
    features = BASE_FEATURES if features is None else features
    kw = dict(labkw); kw.setdefault('dir_feats', ds.feat)
    return label_bars(models, ds.X(features), ds.dates, ds.nifty,
                      ds.trend_raw, ds.n_fit, list(features), **kw)


# ---- THE OFF-VALUE PROOF --------------------------------------------------
print('OFF-VALUE PROOF: R0 (harness + dec_conf(CONF_L)) vs the UNTOUCHED engine')
_t0 = time.time()
for _d in DSETS:
    _ms = _d.fits(BASE_SEED)
    _a = label_untouched(_d, _ms)['tactical_regime_state'].values
    _b = label_decide(_d, _ms, dec_conf(CONF_L))['tactical_regime_state'].values
    assert np.array_equal(_a, _b), (
        f'{_d.name}: the rule harness does NOT reproduce the shipped engine at its '
        f'OFF value -- {100*np.mean(_a != _b):.3f}% of bars differ')
    print(f'  {_d.name:<12} {len(_a)} bars  IDENTICAL  (0 differing bars)')
print(f'ASSERT OK on every dataset ({time.time()-_t0:.1f}s). The override changes '
      'nothing at its OFF value,\nso every difference measured below is caused by '
      'the rule and by nothing else.')

## 4. THE DIAGNOSIS — measured, not assumed

This is the section that decides whether the brief's theory survives. It
decomposes emitted `SIDEWAYS` into its three possible sources:

* **A** — `argmax(bull, bear, side) == side`. The side bucket *genuinely wins*.
* **B** — argmax is `BULL`/`BEAR` but `max(masses) < CONF_L`, so the override
  forces `SIDE`. **This is the absolute-majority tax the brief blames.**
* **C** — what `confirm_delay(CONFIRM_BARS)` adds or removes on top.

`A + B` must equal `dir_raw == SIDE`, and `C` must equal the emitted `SIDEWAYS`
occupancy exactly — both asserted, so the decomposition cannot be a
reconstruction that merely resembles the engine.

In [ ]:
DIAG = {}
for ds in DSETS:
    print('=' * 78)
    print(f'{ds.name}   [{ds.tag}]')
    print('=' * 78)
    ms = ds.fits(BASE_SEED)
    lab = label_untouched(ds, ms)['tactical_regime_state'].values
    occ = {lb: 100.0 * float(np.mean(lab == lb)) for lb in REGIME_LABELS}
    print('  shipped engine occupancy: ' +
          '  '.join(f'{k}={occ[k]:5.1f}%' for k in REGIME_LABELS))

    b, s, r, _ = _emdbm_raw(ms, ds.X(BASE_FEATURES), list(BASE_FEATURES),
                            DIRECTION_EXCLUDE, DIRECTION_MODE, None, None, None)
    M = np.stack([b, r, s], axis=1)
    conf = M.max(axis=1)
    arg = np.array(['BULL', 'BEAR', 'SIDE'])[M.argmax(axis=1)]
    after = np.where(conf < CONF_L, 'SIDE', arg)
    emit = confirm_delay(after, CONFIRM_BARS)

    A = 100.0 * float(np.mean(arg == 'SIDE'))
    B = 100.0 * float(np.mean((conf < CONF_L) & (arg != 'SIDE')))
    AB = 100.0 * float(np.mean(after == 'SIDE'))
    C = 100.0 * float(np.mean(emit == 'SIDE'))
    print()
    print('  WHERE DOES SIDEWAYS COME FROM?')
    print(f'    A  side bucket genuinely wins the argmax        : {A:6.2f}%')
    print(f'    B  argmax directional but max < CONF_L -> SIDE  : {B:6.2f}%'
          '   <== the alleged absolute-majority tax')
    print(f'    A+B  dir_raw == SIDE                            : {AB:6.2f}%')
    print(f'    C  after CONFIRM_BARS={CONFIRM_BARS} -> dir_emit == SIDE      : {C:6.2f}%')
    print(f'    emitted SIDEWAYS label                          : {occ["SIDEWAYS"]:6.2f}%')
    assert abs(A + B - AB) < 1e-9, 'A + B must equal dir_raw SIDE'
    assert abs(C - occ['SIDEWAYS']) < 1e-9, 'dir_emit SIDE must equal emitted SIDEWAYS'
    print('    ASSERT OK: A+B == dir_raw SIDE, and C == emitted SIDEWAYS exactly.')

    print()
    print('  WHY B IS TINY -- the distribution of max(bull, bear, side):')
    for q in (1, 5, 10, 25, 50, 75, 95):
        print(f'    p{q:<3} = {np.percentile(conf, q):.3f}')
    for f in (0.35, 0.40, 0.45, 0.50):
        print(f'    fraction below {f:.2f}: {np.mean(conf < f):.4f}')
    print('    The posterior is close to ONE-HOT. The brief\'s hypothetical')
    print('    bull 0.45 / neutral 0.30 / bear 0.25 bar is not a bar this engine')
    print('    produces at any material rate.')

    print()
    print('  COUNTERFACTUAL -- dir_raw under argmax alone vs argmax + CONF_L:')
    for nm, a in (('pure MAP     ', arg), ('argmax+CONF_L', after)):
        u, c = np.unique(a, return_counts=True)
        print(f'    {nm} ' + '  '.join(f'{x}={100*y/ds.n:6.2f}%' for x, y in zip(u, c)))
    print(f'    -> removing the override entirely moves SIDE by '
          f'{abs(100*np.mean(arg=="SIDE") - 100*np.mean(after=="SIDE")):.2f} pp.')
    DIAG[ds.name] = dict(A=A, B=B, AB=AB, C=C, occ=occ, conf=conf,
                         delta=s - np.maximum(b, r))
    print()

print('=' * 78)
print('VERDICT ON THE BRIEF\'S DIAGNOSIS: REFUTED.')
print('  CONF_L=0.50 is very nearly INERT. It is not what inflates SIDEWAYS.')
print('  SIDEWAYS is high because the SIDE bucket WINS THE ARGMAX -- the single')
print('  middle-ranked state (1 of 5 under direction_buckets: 2 bear / 1 side /')
print('  2 bull) is the MODAL market state and collects far more than 1/5 of bars.')
print('  Any rule that only makes the DIRECTIONAL buckets work harder (R1-R5)')
print('  therefore cannot lower SIDEWAYS -- several of them RAISE it. The lever')
print('  has to act on what SIDE must prove. That is R6, and then R10.')
print('=' * 78)

In [ ]:
# ===========================================================================
# FIGURE 0 -- the diagnosis, in one picture.
# ===========================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
d0 = DIAG[PRIMARY.name]
ax = axes[0]
bars = ax.bar(['A\nside bucket\nwins argmax', 'B\nforced by\nCONF_L=0.50',
               'C\nconfirm_delay\nnet effect'],
              [d0['A'], d0['B'], d0['C'] - d0['AB']],
              color=['#8B0000', '#4477AA', '#999999'])
for b_, v in zip(bars, [d0['A'], d0['B'], d0['C'] - d0['AB']]):
    ax.annotate(f'{v:+.2f} pp' if v < 1 else f'{v:.2f} pp',
                (b_.get_x() + b_.get_width() / 2, v), ha='center',
                va='bottom' if v >= 0 else 'top', fontsize=11, fontweight='bold')
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('contribution to emitted SIDEWAYS (pp)')
ax.set_title(f'Decomposition of SIDEWAYS = {d0["C"]:.1f}%  ({PRIMARY.name})\n'
             'the alleged absolute-majority tax is bar B', fontsize=11)

ax = axes[1]
ax.hist(d0['conf'], bins=60, color='#4477AA', alpha=0.85)
ax.axvline(CONF_L, color='crimson', ls='--', lw=1.8,
           label=f'CONF_L = {CONF_L}  ({100*np.mean(d0["conf"] < CONF_L):.2f}% of bars below)')
ax.set_xlabel('max(bull_mass, bear_mass, side_mass)')
ax.set_ylabel('bars')
ax.set_title('The winning bucket mass is near one-hot,\nso the 0.50 floor almost never binds',
             fontsize=11)
ax.legend(fontsize=9)
fig.suptitle(f'FIG 0 — the brief\'s diagnosis is REFUTED   [{PRIMARY.tag}]',
             fontsize=12, y=1.02)
fig.tight_layout()
_p = f'{FIGDIR}/fig0_diagnosis.png'
fig.savefig(_p, dpi=130, bbox_inches='tight'); FIGS.append(_p)
print('saved', _p)
plt.show()

## 5. Metrics, the ZigZag, the leakage tripwire, and the machinery self-tests

`S`, `L`, `W`, occupancy and guards `G1`–`G4` are reused **verbatim** from
`build_stability_lag.py`. The ZigZag deliberately uses future prices — it is a
*retrospective evaluation* metric, exactly like a forward return — so a
four-part tripwire asserts no ZigZag output can ever reach `label_bars`.

In [ ]:
# ===========================================================================
# METRIC PARAMETERS -- declared before any candidate number exists.
# ===========================================================================
R_SEED_SETS   = 4       # R = 4 DISJOINT seed sets
ZZ_PCT        = 2.0     # ZigZag reversal threshold, %
W_GUARD_MAX   = 25.0    # G3
OCC_MIN_PCT   = 3.0     # G1 / G2
OCC_MAX_PCT   = 50.0    # G1
SIDEWAYS_MAX  = 50.0    # G4

# The acceptance band for this study, from the brief. Declared here, not chosen
# after seeing a table.
SIDE_TARGET_LO, SIDE_TARGET_HI = 18.0, 25.0
SIDE_TARGET_MID = 20.0

# The random-promotion control uses a fixed seed so it is reproducible.
CONTROL_SEED = 20260730

print(f'R={R_SEED_SETS} disjoint seed sets   ZigZag={ZZ_PCT}%   G3 W<={W_GUARD_MAX}   '
      f'G1 occ in [{OCC_MIN_PCT},{OCC_MAX_PCT}]%   G4 SIDEWAYS<={SIDEWAYS_MAX}%')
print(f'acceptance band for SIDEWAYS: [{SIDE_TARGET_LO}, {SIDE_TARGET_HI}]%, '
      f'target {SIDE_TARGET_MID}%')

In [ ]:
# ===========================================================================
# THE ZIGZAG -- LABEL-BLIND, RETROSPECTIVE. Sees prices only; never a label.
# (verbatim from build_stability_lag.py)
# ===========================================================================
ZZ_CALLS = 0
ZZ_IDS = set()
ZZ_ARRAYS = []


def zigzag_pivots(px, pct=ZZ_PCT):
    '''Indices of ZigZag pivots on a close series, `pct`% reversal threshold.

    RETROSPECTIVE BY CONSTRUCTION: a pivot at bar i is only confirmed once price
    has moved pct% away from it, which happens at some bar > i. That is exactly
    the look-ahead this function is allowed to have and `label_bars` is not.'''
    global ZZ_CALLS
    px = np.asarray(px, dtype=float)
    n = len(px)
    if n < 3:
        out = np.array([], dtype=int)
    else:
        hi_i = lo_i = 0
        d, start, piv, ext_i = 0, None, [], 0
        for i in range(1, n):
            if px[i] > px[hi_i]:
                hi_i = i
            if px[i] < px[lo_i]:
                lo_i = i
            if (px[hi_i] - px[i]) / px[hi_i] * 100.0 >= pct:
                d, piv, ext_i, start = -1, [hi_i], i, i
                break
            if (px[i] - px[lo_i]) / px[lo_i] * 100.0 >= pct:
                d, piv, ext_i, start = +1, [lo_i], i, i
                break
        if d == 0:
            out = np.array([], dtype=int)
        else:
            for i in range(start + 1, n):
                if d > 0:
                    if px[i] >= px[ext_i]:
                        ext_i = i
                    elif (px[ext_i] - px[i]) / px[ext_i] * 100.0 >= pct:
                        piv.append(ext_i); d, ext_i = -1, i
                else:
                    if px[i] <= px[ext_i]:
                        ext_i = i
                    elif (px[i] - px[ext_i]) / px[ext_i] * 100.0 >= pct:
                        piv.append(ext_i); d, ext_i = +1, i
            if ext_i != piv[-1]:
                piv.append(ext_i)
            out = np.asarray(piv, dtype=int)
    ZZ_CALLS += 1
    ZZ_IDS.add(id(out))
    ZZ_ARRAYS.append(out)
    return out


def zigzag_swings(px, pct=ZZ_PCT):
    px = np.asarray(px, dtype=float)
    piv = zigzag_pivots(px, pct)
    sw = []
    for a, b in zip(piv[:-1], piv[1:]):
        mv = (px[b] - px[a]) / px[a] * 100.0
        if abs(mv) >= pct:
            sw.append((int(a), int(b), 1 if mv > 0 else -1))
    ZZ_IDS.add(id(sw))
    return sw


# ---- THE LEAKAGE TRIPWIRE. `label_bars` is SHADOWED. ----------------------
_label_bars_raw = label_bars
LABEL_CALLS = 0
LABEL_PHASE_OPEN = True


def label_bars(*args, **kwargs):
    '''Guarded shadow of the engine's label_bars. Adds asserts, changes nothing.'''
    global LABEL_CALLS
    assert ZZ_CALLS == 0 or LABEL_PHASE_OPEN, (
        'LEAKAGE: a label was produced AFTER a ZigZag had been computed.')
    _bw = kwargs.get('bar_dir_weight', BAR_DIR_WEIGHT)
    assert _bw == 0.0, f'BAR_DIR_WEIGHT drifted to {_bw} inside a labelling call'
    for v in list(args) + list(kwargs.values()):
        assert id(v) not in ZZ_IDS, 'LEAKAGE: a ZigZag output was passed to label_bars'
        if isinstance(v, np.ndarray):
            for z in ZZ_ARRAYS:
                assert not np.shares_memory(v, z), \
                    'LEAKAGE: a label_bars argument aliases ZigZag memory'
    LABEL_CALLS += 1
    return _label_bars_raw(*args, **kwargs)


print('ZigZag + leakage tripwire ready.')
print('  check 1  phase ordering  : label_bars asserts ZZ_CALLS == 0')
print('  check 2  object identity : every argument checked against ZZ_IDS')
print('  check 3  memory aliasing : np.shares_memory vs every ZigZag array')
print('  check 4  decree          : bar_dir_weight == 0.0 on every call')

In [ ]:
# ===========================================================================
# S / L / W / occupancy / guards  (verbatim from build_stability_lag.py)
# ===========================================================================
DIR_OF_LABEL = {'H_BULL': 1, 'L_BULL': 1, 'SIDEWAYS': 0, 'L_BEAR': -1, 'H_BEAR': -1}


def emitted_direction(labels):
    return np.array([DIR_OF_LABEL[x] for x in np.asarray(labels)], dtype=int)


def metric_S(label_sets):
    '''S = mean pairwise 5-LABEL agreement (%) across R seed sets.'''
    R = len(label_sets)
    M = np.full((R, R), np.nan)
    vals = []
    for i in range(R):
        M[i, i] = 100.0
        for j in range(i + 1, R):
            a, b = np.asarray(label_sets[i]), np.asarray(label_sets[j])
            assert len(a) == len(b)
            v = 100.0 * float(np.mean(a == b))
            M[i, j] = M[j, i] = v
            vals.append(v)
    return float(np.mean(vals)), M, vals


def metric_L(labels, swings):
    '''Bars from a ZigZag swing START to the first correctly directed emitted
    label. An UNMATCHED swing scores the FULL swing length -- worst case, not a
    skipped row.'''
    d = emitted_direction(labels)
    lags, unmatched = [], 0
    for i0, i1, sgn in swings:
        hit = np.flatnonzero(d[i0:i1 + 1] == sgn)
        if hit.size:
            lags.append(int(hit[0]))
        else:
            lags.append(int(i1 - i0)); unmatched += 1
    if not lags:
        return np.nan, np.nan, np.array([]), 0
    a = np.asarray(lags, dtype=float)
    return float(np.median(a)), float(np.percentile(a, 75)), a, unmatched


def metric_W(labels):
    a = np.asarray(labels)
    return 0.0 if len(a) < 2 else 100.0 * float(np.sum(a[1:] != a[:-1])) / (len(a) - 1)


def occupancy_pct(labels):
    a = np.asarray(labels)
    return {lb: 100.0 * float(np.mean(a == lb)) for lb in REGIME_LABELS}


def evaluate_guards(occ, W):
    g = {}
    hi = [l for l in REGIME_LABELS if occ.get(l, 0.0) > OCC_MAX_PCT]
    lo = [l for l in REGIME_LABELS if occ.get(l, 0.0) < OCC_MIN_PCT]
    g['G1'] = (not hi and not lo, 'ok' if not (hi or lo)
               else f'outside [{OCC_MIN_PCT},{OCC_MAX_PCT}]%: ' + ','.join(sorted(set(hi + lo))))
    g['G2'] = (not lo, 'ok' if not lo else 'collapsed: ' + ','.join(lo))
    g['G3'] = (W <= W_GUARD_MAX, f'W={W:.2f} vs max {W_GUARD_MAX}')
    g['G4'] = (occ.get('SIDEWAYS', 0.0) <= SIDEWAYS_MAX,
               f"SIDEWAYS={occ.get('SIDEWAYS', 0.0):.1f}% vs max {SIDEWAYS_MAX}%")
    return g


def guards_pass(g):
    return all(v[0] for v in g.values())


print('metrics ready: metric_S (pairwise matrix), metric_L (median/p75, unmatched '
      '= full swing\nlength), metric_W, evaluate_guards (G1-G4).')
print('NO composite score is defined anywhere in this notebook -- by design.')

In [ ]:
# ===========================================================================
# THE ANTI-CHEAT BLOCK.
#
# trend_efficiency = |net displacement| / total path travelled over a TRAILING
# window. Causal: bars <= t only. It feeds the INTENSITY (H vs L) axis of the
# engine and NEVER the direction axis, so grading a DIRECTION rule with it is an
# INDEPENDENT test, not a circular one.
# ===========================================================================
def eff_series(ds, win=None):
    return trend_efficiency(ds.nifty.reindex(ds.dates),
                            EFF_WIN if win is None else win).values


def eff_gap(labels, eff):
    '''THE DECLARED HEADLINE METRIC: median eff on directional bars MINUS median
    eff on SIDEWAYS bars. Reported unmodified, shrink and all.'''
    d = emitted_direction(labels)
    a, b = eff[d != 0], eff[d == 0]
    if a.size == 0 or b.size == 0:
        return np.nan, np.nan, np.nan
    return float(np.median(a) - np.median(b)), float(np.median(a)), float(np.median(b))


def eff_auc(labels, eff):
    '''COMPANION 1. P(eff of a random DIRECTIONAL bar > eff of a random SIDEWAYS
    bar), i.e. the Mann-Whitney statistic. A RANK measure, so it is immune to
    the median-of-a-changing-pool artifact that eff_gap suffers when 9pp of bars
    migrate between the two pools. 0.5 = no discrimination at all.'''
    d = emitted_direction(labels)
    a, b = eff[d != 0], eff[d == 0]
    if a.size == 0 or b.size == 0:
        return np.nan
    order = np.argsort(np.concatenate([a, b]), kind='mergesort')
    ranks = np.empty(len(order), dtype=float)
    ranks[order] = np.arange(1, len(order) + 1)
    # average ranks over ties, so the statistic is exact for tied efficiencies
    vals = np.concatenate([a, b])[order]
    i = 0
    while i < len(vals):
        j = i
        while j + 1 < len(vals) and vals[j + 1] == vals[i]:
            j += 1
        if j > i:
            ranks[order[i:j + 1]] = (i + j + 2) / 2.0
        i = j + 1
    return float((ranks[:len(a)].sum() - len(a) * (len(a) + 1) / 2.0) / (len(a) * len(b)))


def eff_ordering(base_labels, cand_labels, eff):
    '''COMPANION 2, THE ORDERING TEST. Median eff of
        core  : directional under BOTH baseline and candidate
        moved : SIDEWAYS under baseline, DIRECTIONAL under the candidate
        kept  : SIDEWAYS under BOTH
    A genuine rule needs core > moved > kept: the bars it promoted must be more
    directional than the bars it left behind. If moved <= kept the candidate is
    relabelling noise, whatever the occupancy says.'''
    db, dc = emitted_direction(base_labels), emitted_direction(cand_labels)
    core, moved, kept = (db != 0) & (dc != 0), (db == 0) & (dc != 0), (db == 0) & (dc == 0)
    med = lambda m: float(np.median(eff[m])) if m.sum() else np.nan
    return med(core), med(moved), med(kept), int(moved.sum()), (
        med(core) > med(moved) > med(kept) if moved.sum() else None)


def random_promotion_control(base_labels, n_promote, eff, seed=CONTROL_SEED):
    '''THE FALSIFICATION CONTROL. Promote n_promote baseline-SIDEWAYS bars AT
    RANDOM to a direction (the direction of the nearest preceding directional
    bar, so the control is not additionally handicapped by nonsense directions).
    Any candidate that cannot beat THIS is relabelling noise by definition.'''
    rng = np.random.default_rng(seed)
    lab = np.asarray(base_labels).copy()
    d = emitted_direction(lab)
    side_idx = np.flatnonzero(d == 0)
    n_promote = int(min(n_promote, len(side_idx)))
    pick = rng.choice(side_idx, size=n_promote, replace=False)
    # nearest preceding directional label, else nearest following
    for i in np.sort(pick):
        j = i - 1
        while j >= 0 and lab[j] == 'SIDEWAYS':
            j -= 1
        if j < 0:
            j = i + 1
            while j < len(lab) and lab[j] == 'SIDEWAYS':
                j += 1
            if j >= len(lab):
                continue
        lab[i] = lab[j]
    return lab


print('anti-cheat ready:')
print('  eff_gap   HEADLINE  median(dir) - median(SIDEWAYS)   [declared in the brief]')
print('  eff_auc   COMPANION rank statistic, immune to pool-composition artifacts')
print('  eff_ordering        core > moved > kept, or the rule is relabelling noise')
print('  random_promotion_control  the floor any real rule must clear')
print('\nINDEPENDENCE: trend_efficiency feeds the INTENSITY axis (EFF_HI gate), never')
print('the DIRECTION axis. Auditing a direction rule with it is therefore not circular.')

In [ ]:
# ===========================================================================
# MACHINERY SELF-TESTS (pre-label phase).
#
# NOTE ON ORDERING: the ZigZag correctness test lives in the EVAL PHASE, not
# here. Calling zigzag_pivots would set ZZ_CALLS > 0 and the label phase's
# ordering assert would -- correctly -- refuse to run. That assert is the
# leakage proof, so the test moved rather than the proof being weakened.
# ===========================================================================
print('TEST 2 -- metric_L scores an UNMATCHED swing as the FULL swing length')
_lab = np.array(['SIDEWAYS'] * 10)
_m, _p75, _a, _un = metric_L(_lab, [(0, 7, 1)])
assert _un == 1 and _a[0] == 7, (_un, _a)
print(f'  all-SIDEWAYS labels, one 7-bar bull swing -> lag {_a[0]:.0f}, '
      f'unmatched {_un}. PASS\n')

print('TEST 3 -- G4 DRIVEN BY A DEGENERATE STUB (the guard must actually bite)')
_ds = PRIMARY
_deg = label_decide(_ds, _ds.fits(BASE_SEED), dec_all_side)['tactical_regime_state'].values
_occ = occupancy_pct(_deg)
print(f'  degenerate all-SIDE stub -> SIDEWAYS = {_occ["SIDEWAYS"]:.1f}%, '
      f'W = {metric_W(_deg):.2f}')
_g = evaluate_guards(_occ, metric_W(_deg))
for k, (ok, why) in _g.items():
    print(f'    {k}: {"PASS" if ok else "FAIL"}  {why}')
assert not _g['G4'][0], 'G4 FAILED TO FIRE on an all-SIDEWAYS labelling'
assert not _g['G1'][0] and not _g['G2'][0], 'G1/G2 should also fire here'
assert not guards_pass(_g)
print('  ASSERT OK: the degenerate arm is VOID. G4 bites, so it is not decorative.\n')

print('TEST 4 -- the anti-cheat rejects a KNOWN-BAD rule')
# A rule that promotes bars at random must show a WORSE AUC than the baseline.
_base = label_untouched(_ds, _ds.fits(BASE_SEED))['tactical_regime_state'].values
_eff = eff_series(_ds)
_n_side = int(np.sum(emitted_direction(_base) == 0))
_ctl = random_promotion_control(_base, int(0.30 * _n_side), _eff)
_auc_b, _auc_c = eff_auc(_base, _eff), eff_auc(_ctl, _eff)
print(f'  baseline AUC {_auc_b:.4f}   random-promotion control AUC {_auc_c:.4f}   '
      f'delta {_auc_c - _auc_b:+.4f}')
assert _auc_c < _auc_b, 'the anti-cheat did NOT punish random promotion -- it has no teeth'
print('  ASSERT OK: random promotion is punished. The audit discriminates.\n')
print('PRE-LABEL-PHASE SELF-TESTS PASSED (TEST 1 runs in the eval phase, by design).')
assert ZZ_CALLS == 0, 'a self-test computed a ZigZag -- ordering would be broken'
print(f'ZZ_CALLS is still {ZZ_CALLS}.')

## 6. The candidate rules

| id | family | what `SIDEWAYS` means under it | can it lower SIDEWAYS? |
|---|---|---|---|
| `R0` | baseline | argmax, forced to `SIDE` when the winner < `CONF_L`=0.50 | — |
| `R1` | MAP | argmax of the three bucket masses, no floor | no — it *is* the baseline |
| `R2`/`R3` | `CONF_L` swept | same, floor ∈ {0.30 … 0.70} | **no** (below ~0.67 the floor is inert; above it, `SIDEWAYS` *rises*) |
| `R4` | original engine | argmax **STATE** by rank + max-state-prob override | no |
| `R5` | top-2 margin | argmax must beat the runner-up by *m* | **no — only raises it** |
| `R6` | **side margin** | `SIDE` must beat the better directional bucket by *m* | **yes** |
| `R7` | dir margin | `\|bull − bear\|` ≤ *m* | yes |
| `R9` | structural | `N_STATES` 5 → 7 under MAP (side bucket still 1 state) | yes |
| `R10` | **side margin, fit-window calibrated** | `R6` with *m* set from the **fit window** at a target rate | **yes, and self-calibrating** |
| `R11` | dir margin, calibrated | `R7` with *m* set the same way | yes |

`R1`–`R5` are run in full even though section 4 already shows they cannot work.
"Cannot" deserves evidence, and the sweep is the evidence.

In [ ]:
# ===========================================================================
# CANDIDATE DECLARATIONS. Nothing runs yet.
# ===========================================================================
def cand(name, fam, param, kind, maker, N=None, note=''):
    return dict(name=name, fam=fam, param=param, kind=kind, maker=maker,
                N=N or BASE_N, note=note)


def build_candidates(ds):
    C = [cand('R0 BASELINE', 'R0', CONF_L, 'decide', lambda: dec_conf(CONF_L),
              note='the shipped rule')]
    C.append(cand('R1 MAP', 'R1', 0.0, 'decide', lambda: dec_conf(0.0),
                  note='pure MAP -- the shipped rule with the floor removed'))
    for f in (0.30, 0.35, 0.40, 0.45, 0.55, 0.60, 0.65, 0.70):
        C.append(cand(f'R3 CONF_L={f:.2f}', 'R3', f, 'decide',
                      (lambda f=f: (lambda: dec_conf(f)))()))
    for f in (0.30, 0.50):
        C.append(cand(f'R4 orig-engine floor={f:.2f}', 'R4', f, 'state',
                      (lambda f=f: (lambda: state_rule_R4(f)))(),
                      note='pre-V1.1 regime_engine_tactical.py rule'))
    for m in (0.05, 0.10, 0.20, 0.30):
        C.append(cand(f'R5 top-margin={m:.2f}', 'R5', m, 'decide',
                      (lambda m=m: (lambda: dec_topmargin(m)))()))
    for m in (0.10, 0.20, 0.30, 0.35, 0.40, 0.50):
        C.append(cand(f'R6 side-margin={m:.2f}', 'R6', m, 'decide',
                      (lambda m=m: (lambda: dec_sidemargin(m)))()))
    for m in (0.10, 0.20, 0.25, 0.30, 0.40):
        C.append(cand(f'R7 dir-margin={m:.2f}', 'R7', m, 'decide',
                      (lambda m=m: (lambda: dec_dirmargin(m)))()))
    for rate in (0.30, 0.25, 0.22, 0.20, 0.18, 0.15):
        C.append(cand(f'R10 side-target={rate:.2f}', 'R10', rate, 'decide',
                      (lambda r=rate, nf=ds.n_fit: (lambda: dec_side_target(r, nf)))()))
    for rate in (0.25, 0.20, 0.15):
        C.append(cand(f'R11 dir-target={rate:.2f}', 'R11', rate, 'decide',
                      (lambda r=rate, nf=ds.n_fit: (lambda: dec_dir_target(r, nf)))()))
    for N in (6, 7, 8):
        C.append(cand(f'R9 N={N} MAP', 'R9', N, 'decide', lambda: dec_conf(0.0), N=N,
                      note='STRUCTURAL: changes the FIT, not just the rule'))
    C.append(cand('R9+R10 N=7 target=0.20', 'R9+R10', 0.20, 'decide',
                  (lambda nf=ds.n_fit: (lambda: dec_side_target(0.20, nf)))(), N=7,
                  note='STRUCTURAL + calibrated rule'))
    return C


_c = build_candidates(PRIMARY)
print(f'{len(_c)} candidates declared per dataset, before any label exists:')
for x in _c:
    print(f"  {x['name']:<26} [{x['fam']:<6}] N={x['N']}  {x['note']}")
_nfit = len({(x['N'],) for x in _c})
print(f"\nfit budget per dataset: {_nfit} distinct N x {R_SEED_SETS} seed sets x "
      f"K={ENSEMBLE_K} = {_nfit * R_SEED_SETS * ENSEMBLE_K} HMM fits "
      f"(rule-only arms reuse fits)")

## 7. LABEL PHASE — every candidate, R=4 disjoint seed sets, **before any ZigZag exists**

This ordering is the leakage proof. When this section finishes, `ZZ_CALLS` is
still `0` and every label in the notebook has already been written.

In [ ]:
# ===========================================================================
# THE LABEL PHASE.
# ===========================================================================
assert ZZ_CALLS == 0, 'a ZigZag was computed before the label phase -- ordering broken'


def seed_sets(K=None, R=R_SEED_SETS, base=None):
    K = ENSEMBLE_K if K is None else K
    base = BASE_SEED if base is None else base
    ss = [list(range(base + r * K, base + r * K + K)) for r in range(R)]
    flat = [s for st in ss for s in st]
    assert len(set(flat)) == len(flat), 'seed sets must be DISJOINT'
    return ss


SEED_SETS = seed_sets()
print(f'R={R_SEED_SETS} DISJOINT seed sets: {SEED_SETS}\n')

LABELS = {}          # (dataset, candidate) -> [labels_seed_set_0 .. R-1]
THRESH = {}          # (dataset, candidate) -> derived threshold, where applicable
CANDS = {}
t0 = time.time()
for ds in DSETS:
    CANDS[ds.name] = build_candidates(ds)
    for c in CANDS[ds.name]:
        labs = []
        for ss in SEED_SETS:
            ms = ds.fits(ss[0], N=c['N'])
            fn = c['maker']()
            if c['kind'] == 'state':
                out = label_state(ds, ms, fn)
            else:
                out = label_decide(ds, ms, fn)
                if hasattr(fn, 'last_thr'):
                    THRESH[(ds.name, c['name'])] = fn.last_thr
            labs.append(out['tactical_regime_state'].values.copy())
        LABELS[(ds.name, c['name'])] = labs
    print(f"  {ds.name:<12} {len(CANDS[ds.name])} candidates labelled x "
          f"{R_SEED_SETS} seed sets   ({time.time()-t0:5.1f}s cumulative)")

LABEL_PHASE_SECS = time.time() - t0
print(f'\nLABEL PHASE COMPLETE in {LABEL_PHASE_SECS:.1f}s   '
      f'{LABEL_CALLS} label_bars calls')
assert ZZ_CALLS == 0, 'LEAKAGE: a ZigZag existed during the label phase'
print('ASSERT OK: ZZ_CALLS == 0 -- every label in this notebook was produced '
      'before any ZigZag was computed.')

In [ ]:
# ===========================================================================
# EVAL PHASE BEGINS. The label phase is closed; the ZigZag may now be computed.
# ===========================================================================
LABEL_PHASE_OPEN = False
print('LABEL PHASE CLOSED. Any label_bars call from here on fails the ordering check.\n')

print('TEST 1 (deferred to here on purpose) -- ZigZag correctness on a hand-built '
      'sawtooth')
_p = np.array([100, 105, 110, 104, 99, 103, 112, 108, 100], dtype=float)
_piv = zigzag_pivots(_p, pct=4.0)
print(f'  prices {_p.tolist()}')
print(f'  pivots {_piv.tolist()} -> values {_p[_piv].tolist()}')
assert _piv[0] == 0 and 2 in _piv and 4 in _piv and 6 in _piv, 'ZigZag missed a pivot'
_swt = zigzag_swings(_p, pct=4.0)
assert all(abs((_p[b] - _p[a]) / _p[a] * 100) >= 4.0 for a, b, _ in _swt)
assert [g for _, _, g in _swt] == [1, -1, 1, -1], f'alternating swings expected, got {_swt}'
print(f'  swings {_swt}  -> alternate in sign, all clear 4%. PASS\n')

SWINGS, EFFS = {}, {}
for ds in DSETS:
    EFFS[ds.name] = eff_series(ds)
    SWINGS[ds.name] = zigzag_swings(ds.close)
    print(f'  {ds.name:<12} {len(SWINGS[ds.name])} ZigZag swings at {ZZ_PCT}%   '
          f'trend efficiency over EFF_WIN={EFF_WIN} bars')
print(f'ZZ_CALLS is now {ZZ_CALLS}.')

In [ ]:
# ===========================================================================
# SCORE EVERY CANDIDATE. Baseline is row 1 of every table. No composite score.
# ===========================================================================
def score_all(ds):
    eff, sw = EFFS[ds.name], SWINGS[ds.name]
    base = LABELS[(ds.name, 'R0 BASELINE')]
    rows = []
    for c in CANDS[ds.name]:
        labs = LABELS[(ds.name, c['name'])]
        occs = [occupancy_pct(l) for l in labs]
        Ws = [metric_W(l) for l in labs]
        gaps = [eff_gap(l, eff)[0] for l in labs]
        aucs = [eff_auc(l, eff) for l in labs]
        ords = [eff_ordering(b, l, eff) for b, l in zip(base, labs)]
        Ls = [metric_L(l, sw) for l in labs]
        # guards evaluated per seed set; a candidate is VOID if ANY seed set fails
        gds = [evaluate_guards(o, w) for o, w in zip(occs, Ws)]
        failed = sorted({k for g in gds for k, v in g.items() if not v[0]})
        side = [o['SIDEWAYS'] for o in occs]
        rows.append(dict(
            rule=c['name'], fam=c['fam'], param=c['param'], N=c['N'],
            SIDE=float(np.mean(side)), SIDEsd=float(np.std(side)),
            SIDEmin=float(np.min(side)), SIDEmax=float(np.max(side)),
            H_BULL=float(np.mean([o['H_BULL'] for o in occs])),
            L_BULL=float(np.mean([o['L_BULL'] for o in occs])),
            L_BEAR=float(np.mean([o['L_BEAR'] for o in occs])),
            H_BEAR=float(np.mean([o['H_BEAR'] for o in occs])),
            S=metric_S(labs)[0],
            W=float(np.mean(Ws)), Wmax=float(np.max(Ws)),
            gap=float(np.nanmean(gaps)), gapsd=float(np.nanstd(gaps)),
            gapmin=float(np.nanmin(gaps)), gapmax=float(np.nanmax(gaps)),
            AUC=float(np.nanmean(aucs)), AUCsd=float(np.nanstd(aucs)),
            e_core=float(np.nanmean([o[0] for o in ords])),
            e_moved=float(np.nanmean([o[1] for o in ords])),
            e_kept=float(np.nanmean([o[2] for o in ords])),
            n_moved=float(np.mean([o[3] for o in ords])),
            order_ok=all(o[4] for o in ords if o[4] is not None) if any(
                o[4] is not None for o in ords) else None,
            Lmed=float(np.mean([x[0] for x in Ls])),
            Lp75=float(np.mean([x[1] for x in Ls])),
            unmatched=float(np.mean([x[3] for x in Ls])),
            guards='PASS' if not failed else ','.join(failed),
            thr=THRESH.get((ds.name, c['name']), np.nan)))
    return pd.DataFrame(rows)


RES = {ds.name: score_all(ds) for ds in DSETS}
print(f'scored {sum(len(v) for v in RES.values())} candidate/dataset combinations.')

In [ ]:
pd.set_option('display.width', 260)
pd.set_option('display.max_columns', 60)

for ds in DSETS:
    df = RES[ds.name]
    b = df.iloc[0]
    print('=' * 200)
    print(f'{ds.name}   [{ds.tag}]')
    print(f'  BASELINE: SIDEWAYS {b.SIDE:.2f}%   gap {b.gap:.4f} +/- {b.gapsd:.4f} '
          f'(seed range {b.gapmin:.4f}..{b.gapmax:.4f})   AUC {b.AUC:.4f} +/- {b.AUCsd:.4f}   '
          f'S {b.S:.2f}   W {b.W:.2f}   L {b.Lmed:.1f}/{b.Lp75:.1f}')
    print('=' * 200)
    cols = ['rule', 'fam', 'param', 'N', 'SIDE', 'SIDEsd', 'H_BULL', 'L_BULL',
            'L_BEAR', 'H_BEAR', 'S', 'W', 'gap', 'AUC', 'e_core', 'e_moved',
            'e_kept', 'order_ok', 'Lmed', 'Lp75', 'guards']
    print(df[cols].to_string(index=False, float_format=lambda x: f'{x:7.3f}'))
    print()

In [ ]:
# ===========================================================================
# THE RANDOM-PROMOTION CONTROL, matched bar-for-bar to each in-band candidate.
# ===========================================================================
CONTROL = {}                 # (dataset, rule) -> control result. Spans ALL datasets.
for ds in DSETS:
    df = RES[ds.name]
    eff = EFFS[ds.name]
    base = LABELS[(ds.name, 'R0 BASELINE')]
    inband = df[(df.SIDE >= SIDE_TARGET_LO) & (df.SIDE <= SIDE_TARGET_HI) &
                (df.guards == 'PASS')]
    print('=' * 118)
    print(f'{ds.name}: RANDOM-PROMOTION CONTROL  '
          f'({len(inband)} in-band, guard-passing candidates)')
    print('=' * 118)
    if not len(inband):
        print('  none in band.\n'); continue
    print(f'  {"candidate":<26} {"SIDE%":>7} {"AUC":>8} {"ctlAUC":>8} {"AUC-ctl":>9} '
          f'{"gap":>8} {"ctlGap":>8} {"gap-ctl":>9}  verdict')
    for _, r in inband.iterrows():
        aucs, gaps, cauc, cgap = [], [], [], []
        for bl, l in zip(base, LABELS[(ds.name, r.rule)]):
            n_mv = int(np.sum((emitted_direction(bl) == 0) & (emitted_direction(l) != 0)))
            ctl = random_promotion_control(bl, n_mv, eff)
            aucs.append(eff_auc(l, eff)); cauc.append(eff_auc(ctl, eff))
            gaps.append(eff_gap(l, eff)[0]); cgap.append(eff_gap(ctl, eff)[0])
        a, ca = np.mean(aucs), np.mean(cauc)
        g, cg = np.nanmean(gaps), np.nanmean(cgap)
        # A control that promoted EVERY baseline-SIDEWAYS bar leaves an empty
        # SIDEWAYS pool, so its statistics are undefined. That is 'no control
        # available', which is NOT the same as 'failed the control'.
        undef = not (np.isfinite(ca) and np.isfinite(cg))
        ok = (not undef) and (a > ca) and (g > cg)
        CONTROL[(ds.name, r.rule)] = dict(auc=a, ctl_auc=ca, gap=g, ctl_gap=cg,
                                          beats=ok, undefined=undef)
        print(f'  {r.rule:<26} {r.SIDE:7.2f} {a:8.4f} {ca:8.4f} {a-ca:+9.4f} '
              f'{g:8.4f} {cg:8.4f} {g-cg:+9.4f}  '
              + ('CONTROL UNDEFINED (it promoted every SIDEWAYS bar)' if undef
                 else ('BEATS CONTROL' if ok else '*** FAILS CONTROL ***')))
    print()

## 8. Figures

In [ ]:
# ===========================================================================
# FIGURE 1 -- SIDEWAYS% vs the swept parameter, per rule family.
#
# TWO PANELS PER DATASET, and they are NOT interchangeable: the x-axis means
# different things in each. Left, x is a THRESHOLD (a floor or a margin) and
# lower SIDEWAYS is to the RIGHT. Right, x is a REQUESTED OCCUPANCY, so the
# diagonal is the ideal and the curve lying ON it is the whole claim.
# ===========================================================================
FAM_LABEL = {'R3': 'R3  CONF_L floor', 'R5': 'R5  top-2 margin',
             'R6': 'R6  side margin (fixed)', 'R7': 'R7  dir margin (fixed)',
             'R10': 'R10 side margin (fit-window target rate)',
             'R11': 'R11 dir margin (fit-window target rate)',
             'R1': 'R1  MAP', 'R4': 'R4  original engine rule',
             'R9': 'R9  N_STATES (structural)',
             'R9+R10': 'R9+R10 N=7 + target rate'}
FAM_COLOR = {'R3': '#999999', 'R5': '#BBAA33', 'R6': '#4477AA', 'R7': '#66CCEE',
             'R10': '#CC3311', 'R11': '#EE7733', 'R1': '#AA3377',
             'R4': '#117733', 'R9': '#882255', 'R9+R10': '#000000'}
THRESH_FAMS, TARGET_FAMS = ['R3', 'R5', 'R6', 'R7'], ['R10', 'R11']

fig, axes = plt.subplots(len(DSETS), 2, figsize=(15, 5.0 * len(DSETS)), squeeze=False)
for row, ds in zip(axes, DSETS):
    df = RES[ds.name]
    b0 = df.iloc[0]
    for ax, fams, xlab in ((row[0], THRESH_FAMS, 'threshold (CONF_L floor / margin)'),
                           (row[1], TARGET_FAMS, 'REQUESTED SIDEWAYS rate')):
        ax.axhspan(SIDE_TARGET_LO, SIDE_TARGET_HI, color='#2ca02c', alpha=0.13, zorder=0)
        ax.axhline(SIDE_TARGET_MID, color='#2ca02c', ls=':', lw=1.4, zorder=1)
        ax.axhline(b0.SIDE, color='k', ls='--', lw=1.6, zorder=2,
                   label=f'R0 baseline = {b0.SIDE:.1f}%')
        for fam in fams:
            sub = df[df.fam == fam].sort_values('param')
            if len(sub):
                ax.plot(sub.param, sub.SIDE, 'o-', color=FAM_COLOR[fam], lw=2.0,
                        ms=6, label=FAM_LABEL[fam])
        ax.set_xlabel(xlab)
        ax.set_ylabel('SIDEWAYS occupancy (%)')
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8, loc='best')
    # the identity line: what a self-calibrating knob is supposed to deliver
    _r = np.array(sorted(df[df.fam == 'R10'].param.unique())) * 100
    row[1].plot(_r / 100, _r, ls='--', lw=1.5, color='#2ca02c', zorder=1,
                label='_nolegend_')
    _dev = float((df[df.fam == 'R10'].SIDE / 100 - df[df.fam == 'R10'].param).abs().mean())
    row[1].annotate('dashed green = "you get what you ask for"\n'
                    f'mean |delivered - requested| = {100*_dev:.1f} pp',
                    xy=(0.97, 0.04), xycoords='axes fraction', ha='right', va='bottom',
                    fontsize=9, bbox=dict(boxstyle='round', fc='#e8f5e9', ec='#2ca02c'))
    row[0].set_title(f'{ds.name} — THRESHOLD knobs\n{ds.tag}', fontsize=9)
    row[1].set_title(f'{ds.name} — TARGET-RATE knobs\n{ds.tag}', fontsize=9)
axes[0][0].annotate("R3 (the brief's knob) is FLAT below ~0.65,\nthen goes the WRONG WAY",
                    xy=(0.97, 0.05), xycoords='axes fraction', ha='right', va='bottom',
                    fontsize=9,
                    bbox=dict(boxstyle='round', fc='#fff3cd', ec='#856404'))
fig.suptitle('FIG 1 — SIDEWAYS% vs the swept parameter. '
             'Green band = the 18-25% acceptance band.', fontsize=12)
fig.tight_layout()
_p = f'{FIGDIR}/fig1_sideways_vs_param.png'
fig.savefig(_p, dpi=130, bbox_inches='tight'); FIGS.append(_p)
print('saved', _p)
print()
print('TRACKING -- how closely a target-rate knob DELIVERS what it is asked for.')
print('The threshold is a FIT-WINDOW quantile, so full-sample occupancy can drift')
print('when the fit window is unrepresentative of the rest of the series. Reported,')
print('not hidden:')
for _o in DSETS:
    _sub = RES[_o.name]
    _sub = _sub[_sub.fam == 'R10']
    _d = (_sub.SIDE / 100 - _sub.param).abs()
    print(f'  {_o.name:<12} mean |delivered - requested| = {100*_d.mean():4.1f} pp   '
          f'max {100*_d.max():4.1f} pp')
print('  -> tight on the REAL series; looser on the synthetic one, whose last 30%')
print('     is a deliberately different regime block from its fit window.')
plt.show()

In [ ]:
# ===========================================================================
# FIGURE 2 -- THE ANTI-CHEAT VIEW. Does the discrimination hold as SIDEWAYS falls?
# ===========================================================================
ds = PRIMARY
df = RES[ds.name]
base = df.iloc[0]
fig, axes = plt.subplots(1, 2, figsize=(15, 5.6))

for ax, (key, sdkey, ylab, ttl) in zip(axes, [
        ('gap', 'gapsd', 'median eff(directional) - median eff(SIDEWAYS)',
         'HEADLINE metric (declared in the brief)'),
        ('AUC', 'AUCsd', 'P(eff of a directional bar > eff of a SIDEWAYS bar)',
         'COMPANION rank metric (immune to pool-composition)')]):
    ax.axvspan(SIDE_TARGET_LO, SIDE_TARGET_HI, color='#2ca02c', alpha=0.13, zorder=0)
    # baseline seed range as a horizontal band -- the headroom check
    lo, hi = (base.gapmin, base.gapmax) if key == 'gap' else \
             (base.AUC - base.AUCsd, base.AUC + base.AUCsd)
    ax.axhspan(lo, hi, color='k', alpha=0.09, zorder=0)
    ax.axhline(base[key], color='k', ls='--', lw=1.5, zorder=2,
               label=f'R0 baseline = {base[key]:.4f}')
    for fam in df.fam.unique():
        if fam == 'R0':
            continue
        sub = df[df.fam == fam]
        ax.errorbar(sub.SIDE, sub[key], yerr=sub[sdkey], fmt='o',
                    ms=6, capsize=2, lw=1,
                    color=FAM_COLOR.get(fam, '#AA3377'), label=fam, alpha=0.9)
    ax.plot(base.SIDE, base[key], marker='*', ms=22, color='k', zorder=6,
            markeredgecolor='w', label='_nolegend_')
    if key == 'AUC':
        ax.axhline(0.5, color='crimson', ls=':', lw=1.4,
                   label='0.5 = NO discrimination')
    ax.set_xlabel('SIDEWAYS occupancy (%)   <-- lower is the goal')
    ax.set_ylabel(ylab)
    ax.set_title(ttl, fontsize=11)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, ncol=2)
axes[0].annotate('grey band = baseline SEED RANGE\n(a point inside it is not a shrink)',
                 xy=(0.03, 0.05), xycoords='axes fraction', fontsize=9,
                 bbox=dict(boxstyle='round', fc='#eeeeee', ec='#666666'))
fig.suptitle(f'FIG 2 — THE ANTI-CHEAT VIEW: does discrimination hold as SIDEWAYS falls?  '
             f'[{ds.name}: {ds.tag}]', fontsize=12)
fig.tight_layout()
_p = f'{FIGDIR}/fig2_anticheat.png'
fig.savefig(_p, dpi=130, bbox_inches='tight'); FIGS.append(_p)
print('saved', _p)
plt.show()

In [ ]:
# ===========================================================================
# PICK THE RECOMMENDATION -- purely mechanically, from the declared criteria.
#   in the 18-25% band, all guards PASS on every seed set, ordering test OK,
#   beats the random-promotion control on BOTH gap and AUC, W within G3.
#   Among survivors, take the one whose AUC is closest to (or above) baseline.
# ===========================================================================
ds = PRIMARY
df = RES[ds.name]
base = df.iloc[0]
elig = df[(df.SIDE >= SIDE_TARGET_LO) & (df.SIDE <= SIDE_TARGET_HI) &
          (df.guards == 'PASS') & (df.order_ok == True) &
          (df.rule != 'R0 BASELINE')].copy()
elig['beats_ctl'] = [CONTROL.get((ds.name, r), {}).get('beats', False) for r in elig.rule]
elig = elig[elig.beats_ctl]

# CROSS-DATASET ROBUSTNESS, applied as a FILTER not a tiebreak. A rule whose
# constant is a GUESSED ABSOLUTE NUMBER is not scale-free: R6/R7 at a fixed
# margin hit ~20% SIDEWAYS here and COLLAPSE the label to ~2% on the synthetic
# series (G2). The synthetic arm exists for exactly this check, so it is
# enforced: a candidate must pass every guard on EVERY dataset it was run on.
_cross = {}
for _o in DSETS:
    for _, _r in RES[_o.name].iterrows():
        _cross.setdefault(_r.rule, []).append((_o.name, _r.guards))
elig['cross'] = [';'.join(f'{n}:{g}' for n, g in _cross[r]) for r in elig.rule]
_bad = elig[[any(g != 'PASS' for _, g in _cross[r]) for r in elig.rule]]
if len(_bad):
    print('DISQUALIFIED ON A NON-PRIMARY DATASET (not scale-free):')
    for _, _r in _bad.iterrows():
        print(f'  {_r.rule:<26} {_r.cross}')
    print()
elig = elig[[all(g == 'PASS' for _, g in _cross[r]) for r in elig.rule]]
elig['auc_delta'] = elig.AUC - base.AUC
# SELECTION RULE, declared: among candidates whose AUC is not materially below
# baseline (within ONE baseline seed sd -- the headroom check), take the one whose
# SIDEWAYS lands CLOSEST TO THE 20% TARGET. The target is the requirement; AUC is
# the guard on it. Nothing here is a composite score: it is a filter, then a
# distance to a number that was declared in section 5 before any result existed.
elig = elig[elig.AUC >= base.AUC - base.AUCsd]
elig['dist_to_target'] = (elig.SIDE - SIDE_TARGET_MID).abs()
# Distance is BUCKETED TO WHOLE PERCENTAGE POINTS before it is used to rank.
# Sub-1pp differences in occupancy are inside the seed spread and must not be
# allowed to decide between two candidates; AUC breaks ties within a bucket.
elig['dist_bucket'] = np.ceil(elig.dist_to_target)
elig = elig.sort_values(['dist_bucket', 'auc_delta'], ascending=[True, False])

print('ELIGIBLE CANDIDATES (in band, guards pass on EVERY seed set, ordering test OK,')
print('beats the random-promotion control, AUC within one baseline seed sd),')
print(f'ranked by |SIDEWAYS - {SIDE_TARGET_MID:.0f}%|:')
print(elig[['rule', 'fam', 'N', 'SIDE', 'SIDEsd', 'S', 'W', 'gap', 'AUC',
            'auc_delta', 'dist_to_target', 'dist_bucket', 'Lmed', 'Lp75']].to_string(
    index=False, float_format=lambda x: f'{x:8.4f}'))

if not len(elig):
    print('\nNO CANDIDATE SATISFIED EVERY CRITERION. Near-misses, so the binding '
          'constraint is visible:')
    nm = df[(df.SIDE >= SIDE_TARGET_LO) & (df.SIDE <= SIDE_TARGET_HI)]
    print(nm[['rule', 'SIDE', 'S', 'W', 'gap', 'AUC', 'e_moved', 'e_kept',
              'order_ok', 'guards']].to_string(index=False,
                                               float_format=lambda x: f'{x:8.4f}'))
RULE_ONLY = elig[elig.N == BASE_N]
assert len(RULE_ONLY), 'no rule-only candidate survived'
WINNER = RULE_ONLY.iloc[0]
STRUCTURAL = elig[elig.N != BASE_N]
RUNNERUP = STRUCTURAL.iloc[0] if len(STRUCTURAL) else None

print(f'\nRULE-ONLY RECOMMENDATION : {WINNER.rule}')
if RUNNERUP is not None:
    print(f'STRUCTURAL ALTERNATIVE   : {RUNNERUP.rule}  '
          '(changes the FIT -- bigger change, reported separately)')

In [ ]:
# ===========================================================================
# FIGURE 3 -- THE REFERENCE CASE. Baseline vs the recommendation, in the
# MASTER's regime-background style: black price line, full-height bands
# (shade_bands / get_xaxis_transform), tight EQUAL y-limits via set_price_ylim,
# NOT zero-anchored.
# ===========================================================================
ds = PRIMARY
REF_LO, REF_HI = pd.Timestamp('2025-02-01'), pd.Timestamp('2025-07-15')
mask = (ds.dates >= REF_LO) & (ds.dates <= REF_HI)
idx = ds.dates[mask]
px = pd.Series(ds.close[mask], index=idx)
print(f'reference window {idx[0]:%Y-%m-%d} -> {idx[-1]:%Y-%m-%d}   {len(idx)} bars')
print(f'  V-bottom {px.min():.0f} on {px.idxmin():%Y-%m-%d}   '
      f'recovery high {px.max():.0f} on {px.idxmax():%Y-%m-%d}   '
      f'advance {100*(px.max()/px.min()-1):+.1f}%')

panels = [('R0 BASELINE', RES[ds.name].iloc[0]), (WINNER.rule, WINNER)]
if RUNNERUP is not None:
    panels.append((RUNNERUP.rule, RUNNERUP))

fig, axes = plt.subplots(len(panels), 1, figsize=(15, 3.4 * len(panels)), sharex=True)
axes = np.atleast_1d(axes)
for ax, (rule, row) in zip(axes, panels):
    lab = pd.Series(LABELS[(ds.name, rule)][0][mask], index=idx)
    # CONTIGUOUS spans. The master's `regime_blocks` ends a block at its LAST
    # BAR, so on DAILY bars every weekend/holiday gap renders as a white
    # unshaded stripe (invisible on 2h bars, conspicuous here). Each block is
    # therefore extended to the START of the next one, and the last to the final
    # bar. shade_bands takes explicit spans, so the harvested helper is USED,
    # not edited.
    _b = regime_blocks(lab)
    _spans = [(_b[k][0], _b[k][1], _b[k + 1][1] if k + 1 < len(_b) else _b[k][2])
              for k in range(len(_b))]
    assert all(a1 <= b0_ for (_, _, a1), (_, b0_, _) in zip(_spans, _spans[1:])), \
        'contiguous spans must not overlap'
    shade_bands(ax, _spans, alpha=0.38)
    ax.plot(idx, px.values, color='black', lw=1.3, zorder=4)
    set_price_ylim(ax, px.values, pad=0.03, tag=f'ref::{rule}')
    occ_w = 100.0 * float((lab == 'SIDEWAYS').mean())
    ax.set_title(f'{rule}   —   SIDEWAYS in this window {occ_w:.1f}%   '
                 f'(full series {row.SIDE:.1f}%)', fontsize=11, loc='left')
    ax.set_ylabel('NIFTY 50')
    ax.grid(alpha=0.2, zorder=0)
# EQUAL y-limits across panels, and assert they exclude 0.
_lo = min(a.get_ylim()[0] for a in axes)
_hi = max(a.get_ylim()[1] for a in axes)
for a in axes:
    a.set_ylim(_lo, _hi)
assert _lo > 0, 'reference-case y-axis is zero-anchored -- the squashed-panel bug'
assert _lo > 0.9 * px.min() and _hi < 1.1 * px.max(), 'reference y-limits are not tight'
print(f'  y-limits {_lo:.0f}..{_hi:.0f}  (price range {px.min():.0f}..{px.max():.0f}) '
      '-> tight, equal across panels, NOT zero-anchored. ASSERT OK')
import matplotlib.patches as _mp
fig.legend(handles=[_mp.Patch(color=REGIME_COLORS[r], alpha=0.7, label=r)
                    for r in REGIME_LABELS],
           loc='lower center', ncol=5, fontsize=10, frameon=False,
           bbox_to_anchor=(0.5, -0.02))
fig.suptitle('FIG 3 — reference case: the 2025 V-bottom and recovery, baseline vs '
             f'recommendation   [{ds.tag}]', fontsize=12)
fig.tight_layout()
_p = f'{FIGDIR}/fig3_reference_case.png'
fig.savefig(_p, dpi=130, bbox_inches='tight'); FIGS.append(_p)
print('saved', _p)
plt.show()

## 9. Iteration history — what was tried, what it gave, why I moved on

| # | tried | result | why I moved on |
|---|---|---|---|
| 1 | **`R1` MAP / `R3` `CONF_L` swept 0.30–0.50** — the brief's own hypothesis | `SIDEWAYS` moves by **0.05 pp**. `CONF_L` binds on 0.1% of bars. | The diagnosis is refuted. The rule is *already* MAP; there is no absolute-majority tax to remove. |
| 2 | **`R3` `CONF_L` raised to 0.55–0.70** | `SIDEWAYS` **rises** to 31% → 47%. | Wrong direction, and it exposes a mass cliff near ⅔ (two states sharing a bucket), which is why the floor is inert below it. |
| 3 | **`R5` top-2 margin** | `SIDEWAYS` only **rises**. | A downgrade-only rule cannot lower the label it downgrades *to*. Structurally incapable. |
| 4 | **`R4` original pre-V1.1 engine rule** | comparable `SIDEWAYS`, no improvement. | The old rule's "1 of 5 states is SIDEWAYS" intuition is already what `direction_buckets` implements — and the middle state is simply the modal state. |
| 5 | **`R6` fixed side-margin** — first rule that makes `SIDE` earn the label | *m*=0.40 → **20.4%** on the real series. Guards pass, `W` +0.8. | Works, but **not scale-free**: the same *m*=0.40 gives **1.5%** `SIDEWAYS` on the synthetic series — a `G2` collapse. A fixed absolute margin is a guessed constant, not a knob. |
| 6 | **`R7` fixed dir-margin** | similar to `R6`, same scale problem. | Same objection. |
| 7 | **`R10` side-margin at a FIT-WINDOW TARGET RATE** | ≈**20.6%** on the real series with seed sd **0.22 pp** (baseline 0.90), and it *raises* the synthetic series' too-low `SIDEWAYS` instead of destroying it. | **This is the recommendation.** It re-uses the engine's own `H_TARGET_RATE` device, is strictly causal, and expresses the requirement ("`SIDEWAYS` ≈ 20%") directly as the knob instead of via a guessed constant. |
| 8 | **`R9` `N_STATES` 5 → 6 / 7 / 8** (structural) | `N=6` **fails the ordering test**; `N=8` misses the band (`n_side` becomes 2); `N=7` hits ~22% **and raises the gap above baseline**. | Kept as the **structural alternative** — strictly better on the anti-cheat, but it changes the *fit*, costs `S`, and is a much bigger change than a decision rule. |

The honest note on the headline metric: the median gap **does** shrink by ~0.015
for every in-band rule-only candidate — slightly outside the baseline's own seed
range. Section 7 shows why: the promoted bars sit *between* the two medians by
construction, so they pull the directional median down. The rank-based **AUC is
flat**, the **ordering test passes**, and every recommended candidate **beats the
random-promotion control on both metrics**. The verdict is that discrimination is
intact and the median shrink is arithmetic — but the shrink is reported, not
explained away, and if the supervisor wants the strict "gap must not shrink"
reading enforced, the answer is `R9 N=7`, which is the only arm whose gap
actually *grows*.

In [ ]:
# ===========================================================================
# 10. CONFIG HAND-OFF -- pasteable Python for the winner.
# ===========================================================================
ds = PRIMARY
w = WINNER
thr = RES[ds.name].set_index('rule').loc[w.rule, 'thr']
print('=' * 78)
print('CONFIG HAND-OFF BLOCK -- paste into the master notebook\'s constants cell')
print('=' * 78)
print(f'''
# ---------------------------------------------------------------------------
# SIDEWAYS FIX  ({w.rule})
#
# Measured on {ds.name}: SIDEWAYS {base.SIDE:.1f}% -> {w.SIDE:.1f}%
#   (per-seed {w.SIDEmin:.1f}..{w.SIDEmax:.1f}%, sd {w.SIDEsd:.2f}pp vs baseline sd {base.SIDEsd:.2f}pp)
#   S  {base.S:.2f} -> {w.S:.2f}      W  {base.W:.2f} -> {w.W:.2f}  (G3 max {W_GUARD_MAX})
#   L  {base.Lmed:.1f}/{base.Lp75:.1f} -> {w.Lmed:.1f}/{w.Lp75:.1f} (median/p75)
#   eff gap {base.gap:.4f} -> {w.gap:.4f}   eff AUC {base.AUC:.4f} -> {w.AUC:.4f}
#   guards: {w.guards}   ordering test: {"PASS" if w.order_ok else "FAIL"}
#
# {ds.tag}
# THE KAGGLE RUN ON THE REAL 2h SERIES REMAINS THE SOURCE OF TRUTH.
# ---------------------------------------------------------------------------
SIDE_RULE        = 'side_target'   # was: implicit MAP + CONF_L floor
SIDE_TARGET_RATE = {w.param:.2f}          # target SIDEWAYS occupancy, calibrated on the FIT WINDOW
CONF_L           = 0.0            # the old floor is INERT (it bound on 0.1% of bars);
                                  # pinned to 0 so the target-rate rule is the only gate

# Drop-in replacement for the two direction lines inside `label_bars`.
# Everything else in label_bars is UNCHANGED.
#
#   OLD:
#     regime_confidence = np.maximum(np.maximum(bull_mass, bear_mass), side_mass)
#     masses  = np.stack([bull_mass, bear_mass, side_mass], axis=1)
#     winner  = masses.argmax(axis=1)
#     dir_raw = np.select([winner == 0, winner == 1, winner == 2],
#                         ['BULL', 'BEAR', 'SIDE'], default='SIDE')
#     dir_raw = np.where(regime_confidence < CONF_L, 'SIDE', dir_raw)
#
#   NEW:
#     delta   = side_mass - np.maximum(bull_mass, bear_mass)
#     # FIT-WINDOW constant. Causal: bars [0, n_fit) only -- exactly the bars the
#     # HMM was trained on. Same device as H_TARGET_RATE.
#     side_thr = float(np.quantile(delta[:n_fit], 1.0 - SIDE_TARGET_RATE))
#     dir_raw = np.where(delta > side_thr, 'SIDE',
#                        np.where(bull_mass >= bear_mass, 'BULL', 'BEAR'))
#     regime_confidence = np.maximum(np.maximum(bull_mass, bear_mass), side_mass)
#
# On this dataset the derived threshold was side_thr = {thr:.4f}
# (it is DERIVED per fit, never hard-coded -- that is the whole point).
''')
if RUNNERUP is not None:
    print(f'''
# ---------------------------------------------------------------------------
# STRUCTURAL ALTERNATIVE, offered but NOT the primary recommendation:
#   {RUNNERUP.rule}
#   N_STATES {BASE_N} -> {int(RUNNERUP.N)}   SIDEWAYS {RUNNERUP.SIDE:.1f}%
#   eff gap {base.gap:.4f} -> {RUNNERUP.gap:.4f}  (the ONLY arm whose gap GROWS)
#   eff AUC {base.AUC:.4f} -> {RUNNERUP.AUC:.4f}
#   COST: S {base.S:.2f} -> {RUNNERUP.S:.2f}, and it changes the FIT, so the
#         config lottery / rows-per-parameter work would have to be redone.
# ---------------------------------------------------------------------------''')

In [ ]:
# ===========================================================================
# 11. FINAL VERDICT + run report.
# ===========================================================================
print('=' * 78)
print('VERDICT')
print('=' * 78)
print("1. THE BRIEF'S DIAGNOSIS IS REFUTED. CONF_L=0.50 forces SIDEWAYS on "
      f"{DIAG[PRIMARY.name]['B']:.2f}% of bars.")
print('   Removing it entirely moves SIDEWAYS by '
      f"{abs(RES[PRIMARY.name].iloc[1].SIDE - base.SIDE):.2f} pp. The rule is already MAP.")
print('   SIDEWAYS is high because the SIDE bucket WINS THE ARGMAX on '
      f"{DIAG[PRIMARY.name]['A']:.1f}% of bars.")
print()
print(f'2. RECOMMENDATION: {WINNER.rule}')
print(f'   SIDEWAYS {base.SIDE:.2f}% -> {WINNER.SIDE:.2f}%  (band '
      f'[{SIDE_TARGET_LO},{SIDE_TARGET_HI}]%, per-seed '
      f'{WINNER.SIDEmin:.1f}..{WINNER.SIDEmax:.1f}%)')
print(f'   S {base.S:.2f} -> {WINNER.S:.2f}   W {base.W:.2f} -> {WINNER.W:.2f} '
      f'(G3 max {W_GUARD_MAX}, max over seeds {WINNER.Wmax:.2f})')
print(f'   L median {base.Lmed:.1f} -> {WINNER.Lmed:.1f}, p75 {base.Lp75:.1f} -> '
      f'{WINNER.Lp75:.1f}   guards {WINNER.guards}')
print(f'   ANTI-CHEAT: gap {base.gap:.4f} -> {WINNER.gap:.4f}  '
      f'(baseline seed range {base.gapmin:.4f}..{base.gapmax:.4f})')
print(f'               AUC {base.AUC:.4f} -> {WINNER.AUC:.4f}  '
      f'(baseline sd {base.AUCsd:.4f}) -> discrimination INTACT')
print(f'               ordering core {WINNER.e_core:.4f} > moved {WINNER.e_moved:.4f} '
      f'> kept {WINNER.e_kept:.4f}  -> '
      f'{"PASS" if WINNER.order_ok else "FAIL"}')
_c = CONTROL.get((PRIMARY.name, WINNER.rule), {})
if _c:
    print(f'               beats random-promotion control: AUC {_c["auc"]:.4f} vs '
          f'{_c["ctl_auc"]:.4f}, gap {_c["gap"]:.4f} vs {_c["ctl_gap"]:.4f}')
print()
print('3. HONEST CAVEAT: the median gap shrinks slightly for every in-band')
print('   rule-only candidate. The rank-based AUC does NOT, the ordering test')
print('   passes, and the control is beaten. If the strict "gap must not shrink"')
if RUNNERUP is not None:
    print(f'   reading is enforced, the answer is {RUNNERUP.rule} '
          f'(gap {base.gap:.4f} -> {RUNNERUP.gap:.4f}),')
    print(f'   at a cost of {base.S - RUNNERUP.S:.2f}pp of S and a changed FIT.')
print()
print('4. DATA: ' + PRIMARY.tag)
print('   Daily bars, not 2h. Absolute occupancy will not match the Kaggle run;')
print('   every comparison here is against the baseline measured on the SAME data.')
print('   THE KAGGLE RUN ON THE REAL 2h SERIES REMAINS THE SOURCE OF TRUTH.')
print()
print('=' * 78)
print('RUN REPORT')
print('=' * 78)
print(f'  datasets            : {[d.name for d in DSETS]}')
print(f'  candidates/dataset  : {len(CANDS[PRIMARY.name])}')
print(f'  seed sets           : {R_SEED_SETS} disjoint -> {SEED_SETS}')
print(f'  label_bars calls    : {LABEL_CALLS}')
print(f'  ZigZag calls        : {ZZ_CALLS}  (all AFTER the label phase closed)')
print(f'  label phase runtime : {LABEL_PHASE_SECS:.1f}s')
print(f'  figures written     : {len(FIGS)}')
for p in FIGS:
    print(f'      {p}  ({os.path.getsize(p)/1024:.0f} KB)')
print(f'  ylim checks recorded: {len(YLIM_CHECKS)}')
for tag, ax, lo, hi in YLIM_CHECKS:
    y0, y1 = ax.get_ylim()
    assert y0 <= lo and y1 >= hi and y0 > 0, f'ylim check failed for {tag}'
print('  ASSERT OK: every shaded price panel brackets its own series and excludes 0.')